In [ ]:
# # =============================================================================
# #  GMM-TDNN Hybrid ASR – Bahasa Jawa  [VERSI PERBAIKAN v2]
# #
# #  PERBAIKAN DARI VERSI SEBELUMNYA:
# #
# #  [FIX-A] CRITICAL: Data leakage di decoder dihapus.
# #          dekode_audio tidak lagi menerima kata_ref_list sebagai petunjuk.
# #          ASR sekarang benar-benar bebas: fitur audio → hipotesis kata.
# #          Decoder mencari kombinasi kata terbaik dari seluruh vocab.
# #
# #  [FIX-B] CRITICAL: Arsitektur TDNN diubah ke CTC (sequence-to-sequence).
# #          Model menerima sekuens frame dan memprediksi sekuens label
# #          menggunakan CTC loss, bukan klasifikasi per-segmen per-kata.
# #
# #  [FIX-C] CRITICAL: Bug logika DP alignment di pecah_frame_gmm diperbaiki.
# #          Backpointer sekarang menyimpan arah (0=stay, 1=advance) bukan
# #          indeks kata, sehingga traceback menghasilkan path monotonic yang benar.
# #
# #  [FIX-D] CRITICAL: Beam search diimplementasi ulang sebagai CTC beam search
# #          yang valid: ekspansi berdasarkan distribusi posterior TDNN per frame,
# #          bukan per posisi kata referensi.
# #
# #  [FIX-E] GMM log_prob_per_frame divektorisasi (hapus loop per komponen).
# #
# #  [FIX-F] collate_fn menggunakan torch.nn.utils.rnn.pad_sequence (lebih efisien).
# #
# #  [FIX-G] Delta manual (_mfcc_manual) menggunakan operasi array numpy (bukan
# #          loop Python), kompleksitas tetap O(T×W) tapi jauh lebih cepat.
# #
# #  [FIX-H] Magic numbers dikonsolidasi menjadi konstanta bernama.
# #
# #  [FIX-I] normalisasi di-cache dengan lru_cache untuk string yang sama.
# #
# #  [FIX-J] except Exception dibatasi hanya pada I/O; bug programmer dibiarkan
# #          muncul (tidak ditelan).
# #
# #  [FIX-K] Checkpoint: setelah early stopping, best model dimuat kembali dengan
# #          eksplisit + validasi bahwa file tidak korup sebelum training baru.
# # =============================================================================

# SAMPLED_ROOT = r"D:\doc\2026\FINAL_CODE\data\sampled_small"

# CSV_PATH = (
#     r"D:\doc\2026\FINAL_CODE\src\codeStat\preModelling"
#     r"\samplingStratified\out_samplingStratified\dataSampledSmall.csv"
# )

# DIR_SAVE = (
#     r"D:\doc\2026\FINAL_CODE\src\codeStat\modelling"
#     r"\voiceModelling\srcVoiceModelling\hybrid\src\GMM-TDNN\out"
# )

# # ── Hyperparameter GMM ────────────────────────────────────────────────────────
# N_COMPONENTS_GMM = 16
# N_ITER_GMM       = 100
# TOL_GMM          = 1e-4

# # ── Hyperparameter TDNN ───────────────────────────────────────────────────────
# TDNN_HIDDEN      = 512
# TDNN_LAYERS      = 5
# TDNN_DROPOUT     = 0.1
# TDNN_EPOCHS      = 30
# TDNN_LR          = 1e-3
# TDNN_BATCH_SIZE  = 16        # CTC butuh sequence panjang, batch lebih kecil
# TDNN_CONTEXTS    = [
#     [-1, 0, 1],
#     [-1, 0, 1],
#     [-3, 0, 3],
#     [-3, 0, 3],
#     [-6, 0, 6],
# ]
# TDNN_PATIENCE    = 5

# # ── Language model ────────────────────────────────────────────────────────────
# N_GRAM           = 3
# BEAM_WIDTH       = 5
# VAL_RATIO        = 0.2

# # ── [FIX-H] Konstanta numerik bernama ────────────────────────────────────────
# LOG_ZERO         = -1e15    # representasi log(0) yang aman
# LOG_EPS          = 1e-300   # epsilon untuk stabilitas log
# COV_REG          = 1e-6     # regularisasi kovarians GMM
# FEAT_EPS         = 1e-8     # epsilon untuk CMVN

# # =============================================================================
# #  IMPORT
# # =============================================================================
# import os, re, glob, math, time, random
# from functools import lru_cache
# import numpy as np
# import pandas as pd
# from collections import defaultdict
# from typing import Dict, List, Tuple, Optional

# try:
#     import torch
#     import torch.nn as nn
#     import torch.nn.functional as F
#     from torch.utils.data import Dataset, DataLoader
#     from torch.nn.utils.rnn import pad_sequence
#     _TORCH = True
# except ImportError:
#     _TORCH = False
#     raise ImportError("PyTorch diperlukan: pip install torch")

# try:
#     import librosa
#     _LIBROSA = True
# except ImportError:
#     _LIBROSA = False

# try:
#     from scipy.io import wavfile as _wavfile
#     _SCIPY = True
# except ImportError:
#     _SCIPY = False


# # =============================================================================
# #  FONEM & NORMALISASI
# # =============================================================================
# PETA_FONEM: Dict[str, List[str]] = {
#     "ng": ["ng"], "ny": ["ny"], "th": ["th"], "dh": ["dh"],
#     "a": ["a"],  "b": ["b"],  "c": ["ch"], "d": ["d"],
#     "e": ["e"],  "f": ["f"],  "g": ["g"],  "h": ["h"],
#     "i": ["i"],  "j": ["j"],  "k": ["k"],  "l": ["l"],
#     "m": ["m"],  "n": ["n"],  "o": ["o"],  "p": ["p"],
#     "r": ["r"],  "s": ["s"],  "t": ["t"],  "u": ["u"],
#     "v": ["v"],  "w": ["w"],  "y": ["y"],  "z": ["z"],
# }

# # [FIX-I] Cache normalisasi agar string yang sama tidak diproses ulang
# @lru_cache(maxsize=4096)
# def normalisasi(teks: str) -> str:
#     teks = teks.lower().strip()
#     teks = re.sub(r"[^a-z\s]", "", teks)
#     return re.sub(r"\s+", " ", teks).strip()

# def kata_ke_fonem(kata: str) -> List[str]:
#     fonem, i = [], 0
#     while i < len(kata):
#         if i + 1 < len(kata) and kata[i: i + 2] in PETA_FONEM:
#             fonem.extend(PETA_FONEM[kata[i: i + 2]]); i += 2
#         elif kata[i] in PETA_FONEM:
#             fonem.extend(PETA_FONEM[kata[i]]); i += 1
#         else:
#             i += 1
#     return fonem if fonem else [kata]

# def buat_lexicon(transkrip_list: List[str]) -> Dict[str, List[str]]:
#     lexicon: Dict[str, List[str]] = {}
#     for kalimat in transkrip_list:
#         for kata in normalisasi(kalimat).split():
#             if kata not in lexicon:
#                 lexicon[kata] = kata_ke_fonem(kata)
#     return lexicon


# # =============================================================================
# #  baca_transkrip_csv (robust, dari versi sebelumnya)
# # =============================================================================
# def baca_transkrip_csv(csv_path: str) -> Dict[str, str]:
#     if not os.path.isfile(csv_path):
#         raise FileNotFoundError(f"CSV tidak ditemukan: {csv_path}")

#     df = None
#     for enc in ("utf-8", "utf-8-sig", "latin-1", "cp1252"):
#         try:
#             df = pd.read_csv(csv_path, encoding=enc)
#             break
#         except (UnicodeDecodeError, pd.errors.ParserError):
#             continue
#     if df is None:
#         raise ValueError(f"Gagal membaca CSV: {csv_path}")

#     df.columns = [c.strip() for c in df.columns]

#     col_path_cands = ["Path_Data", "path_data", "path", "audio_path",
#                       "filename", "file", "wav_path"]
#     col_text_cands = ["sentences", "sentence", "text", "transcription",
#                       "label", "transcript"]

#     col_path = next((c for c in col_path_cands if c in df.columns), None)
#     col_text = next((c for c in col_text_cands if c in df.columns), None)

#     if col_path is None:
#         raise ValueError(f"Kolom path tidak ditemukan. Kolom: {list(df.columns)}")
#     if col_text is None:
#         raise ValueError(f"Kolom transkrip tidak ditemukan. Kolom: {list(df.columns)}")

#     result = {}
#     for _, row in df.iterrows():
#         path_val = str(row[col_path]).strip()
#         text_val = str(row[col_text]).strip()
#         if path_val in ("nan", "") or text_val in ("nan", ""):
#             continue
#         audio_id = os.path.splitext(os.path.basename(path_val))[0]
#         result[audio_id] = text_val.lower()

#     print(f"  CSV dibaca: {len(result)} entri valid dari {len(df)} baris")
#     return result


# # =============================================================================
# #  CMVN
# # =============================================================================
# def apply_cmvn(features: np.ndarray) -> np.ndarray:
#     mean = features.mean(axis=0, keepdims=True)
#     std  = features.std(axis=0, keepdims=True) + FEAT_EPS
#     return (features - mean) / std


# # =============================================================================
# #  EKSTRAKSI FITUR MFCC + CMVN
# # =============================================================================
# _SR         = 16000
# _N_MFCC     = 13
# _N_FFT      = 512
# _HOP_LENGTH = 160
# _WIN_LENGTH = 400
# _N_MELS     = 26
# _PRE_EMPH   = 0.97
# _FMIN       = 0.0
# _FMAX       = 4000.0
# _DELTA_W    = 2
# FEAT_DIM    = _N_MFCC * 3   # 39

# def extract_features(wav_path: str, cmvn: bool = True) -> np.ndarray:
#     if not os.path.isfile(wav_path):
#         raise FileNotFoundError(f"File tidak ditemukan: {wav_path}")
#     if _LIBROSA:
#         feat = _mfcc_librosa(wav_path)
#     elif _SCIPY:
#         feat = _mfcc_manual(wav_path)
#     else:
#         raise ImportError("Install librosa atau scipy")
#     if cmvn:
#         feat = apply_cmvn(feat)
#     return feat

# def _mfcc_librosa(wav_path: str) -> np.ndarray:
#     y, _ = librosa.load(wav_path, sr=_SR, mono=True)
#     if len(y) == 0:
#         raise ValueError(f"Audio kosong: {wav_path}")
#     y      = np.append(y[0], y[1:] - _PRE_EMPH * y[:-1])
#     mfcc   = librosa.feature.mfcc(
#         y=y, sr=_SR, n_mfcc=_N_MFCC,
#         n_fft=_N_FFT, hop_length=_HOP_LENGTH,
#         win_length=_WIN_LENGTH, n_mels=_N_MELS,
#         fmin=_FMIN, fmax=_FMAX, window="hamming",
#     )
#     delta  = librosa.feature.delta(mfcc, width=_DELTA_W * 2 + 1)
#     delta2 = librosa.feature.delta(mfcc, order=2, width=_DELTA_W * 2 + 1)
#     return np.vstack([mfcc, delta, delta2]).T.astype(np.float32)

# def _delta_numpy(feat: np.ndarray, w: int = _DELTA_W) -> np.ndarray:
#     """
#     [FIX-G] Komputasi delta menggunakan operasi array numpy.
#     Menggantikan double loop Python yang O(T*W) dengan operasi vektor.
#     """
#     T = feat.shape[0]
#     denom  = 2.0 * sum(n * n for n in range(1, w + 1))
#     # Padding tepi dengan replikasi
#     padded = np.pad(feat, ((w, w), (0, 0)), mode="edge")
#     delta  = np.zeros_like(feat)
#     for n in range(1, w + 1):
#         delta += n * (padded[w + n: w + n + T] - padded[w - n: w - n + T])
#     return delta / denom

# def _mfcc_manual(wav_path: str) -> np.ndarray:
#     sr, audio = _wavfile.read(wav_path)
#     if audio.dtype == np.int16:
#         audio = audio.astype(np.float32) / 32768.0
#     elif audio.dtype == np.int32:
#         audio = audio.astype(np.float32) / 2147483648.0
#     else:
#         audio = audio.astype(np.float32)
#     if audio.ndim > 1:
#         audio = audio.mean(axis=1)
#     if sr != _SR:
#         n_new = int(len(audio) * _SR / sr)
#         audio = np.interp(
#             np.linspace(0, len(audio) - 1, n_new),
#             np.arange(len(audio)), audio,
#         )
#     y        = np.append(audio[0], audio[1:] - _PRE_EMPH * audio[:-1])
#     n_frames = max(1, 1 + (len(y) - _WIN_LENGTH) // _HOP_LENGTH)
#     # Vectorized frame extraction
#     idx      = np.arange(_WIN_LENGTH)[None, :] + _HOP_LENGTH * np.arange(n_frames)[:, None]
#     idx      = np.clip(idx, 0, len(y) - 1)
#     frames   = y[idx].astype(np.float32)
#     frames  *= np.hamming(_WIN_LENGTH).astype(np.float32)
#     fft_mag  = np.abs(np.fft.rfft(frames, n=_N_FFT))
#     def hz2mel(f): return 1127.0 * np.log1p(f / 700.0)
#     def mel2hz(m): return 700.0 * (np.exp(m / 1127.0) - 1.0)
#     mel_pts = np.linspace(hz2mel(_FMIN), hz2mel(_FMAX), _N_MELS + 2)
#     hz_pts  = mel2hz(mel_pts)
#     bins    = np.floor((_N_FFT + 1) * hz_pts / _SR).astype(int)
#     fb      = np.zeros((_N_MELS, _N_FFT // 2 + 1), dtype=np.float32)
#     for m in range(1, _N_MELS + 1):
#         for k in range(bins[m - 1], bins[m]):
#             fb[m - 1, k] = (k - bins[m - 1]) / max(bins[m] - bins[m - 1], 1)
#         for k in range(bins[m], bins[m + 1]):
#             fb[m - 1, k] = (bins[m + 1] - k) / max(bins[m + 1] - bins[m], 1)
#     mel_e = np.maximum(fft_mag ** 2 @ fb.T, 1e-10)
#     log_e = np.log10(mel_e)
#     N     = _N_MELS
#     dct_m = np.cos(
#         np.pi * np.arange(_N_MFCC)[:, None]
#         * (np.arange(N) + 0.5)[None, :] / N
#     )
#     mfcc = (log_e @ dct_m.T) * np.sqrt(2.0 / N)
#     # [FIX-G] Gunakan delta numpy, bukan loop Python
#     d1 = _delta_numpy(mfcc)
#     d2 = _delta_numpy(d1)
#     return np.concatenate([mfcc, d1, d2], axis=1).astype(np.float32)


# # =============================================================================
# #  GMM (aligner)
# # =============================================================================
# class GaussianMixtureModel:
#     def __init__(self, n_components=4, n_iter=100, tol=1e-3,
#                  reg_covar=COV_REG, random_state=None):
#         self.n_components = n_components
#         self.n_iter       = n_iter
#         self.tol          = tol
#         self.reg_covar    = reg_covar
#         self.random_state = random_state
#         self.weights_     = None
#         self.means_       = None
#         self.covariances_ = None
#         self.converged_   = False
#         self.n_iter_done_ = 0
#         self.log_likelihood_history_ = []

#     def _init_params(self, X):
#         rng  = np.random.default_rng(self.random_state)
#         T, D = X.shape
#         K    = self.n_components
#         idx  = rng.choice(T, size=K, replace=False)
#         self.means_       = X[idx].copy()
#         self.weights_     = np.full(K, 1.0 / K)
#         self.covariances_ = np.tile(np.var(X, axis=0) + self.reg_covar, (K, 1))

#     def _log_gaussian_all(self, X: np.ndarray) -> np.ndarray:
#         """
#         [FIX-E] Hitung log-Gaussian untuk SEMUA komponen sekaligus.
#         Menggantikan loop `for k in range(K)` dengan operasi array.
#         X: (T, D) → return (T, K)
#         """
#         T, D = X.shape
#         K    = self.n_components
#         # mu:  (K, D), var: (K, D)
#         mu   = self.means_           # (K, D)
#         var  = self.covariances_     # (K, D)
#         log_det = np.sum(np.log(var), axis=1)   # (K,)
#         # diff: (T, K, D)
#         diff = X[:, None, :] - mu[None, :, :]
#         maha = np.sum(diff ** 2 / var[None, :, :], axis=2)  # (T, K)
#         log_norm = -0.5 * (D * np.log(2 * np.pi) + log_det)  # (K,)
#         return log_norm[None, :] - 0.5 * maha                 # (T, K)

#     def _e_step(self, X):
#         log_w = np.log(self.weights_ + LOG_EPS) + self._log_gaussian_all(X)
#         lse   = np.max(log_w, axis=1, keepdims=True)
#         log_sum  = lse.squeeze() + np.log(
#             np.sum(np.exp(log_w - lse), axis=1)
#         )
#         log_resp = log_w - log_sum[:, None]
#         return log_resp, log_sum

#     def _m_step(self, X, log_resp):
#         T, D = X.shape
#         K    = self.n_components
#         resp = np.exp(log_resp)
#         Nk   = resp.sum(axis=0) + LOG_EPS
#         self.weights_ = Nk / T
#         self.means_   = (resp.T @ X) / Nk[:, None]
#         diff = X[:, None, :] - self.means_[None, :, :]   # (T, K, D)
#         new_cov = np.sum(
#             resp[:, :, None] * diff ** 2, axis=0
#         ) / Nk[:, None]                                    # (K, D)
#         self.covariances_ = new_cov + self.reg_covar

#     def fit(self, X):
#         assert X.ndim == 2
#         self.n_components = min(self.n_components, X.shape[0])
#         self._init_params(X)
#         self.log_likelihood_history_ = []
#         prev_ll = -np.inf
#         for it in range(self.n_iter):
#             log_resp, log_sum = self._e_step(X)
#             ll = float(log_sum.mean())
#             self.log_likelihood_history_.append(ll)
#             if abs(ll - prev_ll) < self.tol:
#                 self.converged_   = True
#                 self.n_iter_done_ = it + 1
#                 break
#             prev_ll = ll
#             self._m_step(X, log_resp)
#         else:
#             self.converged_   = False
#             self.n_iter_done_ = self.n_iter
#         return self

#     def score(self, X):
#         assert X.ndim == 2
#         _, log_sum = self._e_step(X)
#         return float(log_sum.mean())

#     def log_prob_per_frame(self, X: np.ndarray) -> np.ndarray:
#         """[FIX-E] Satu panggilan _log_gaussian_all, tanpa loop per komponen."""
#         assert X.ndim == 2
#         log_w = np.log(self.weights_ + LOG_EPS) + self._log_gaussian_all(X)
#         lse   = np.max(log_w, axis=1, keepdims=True)
#         return (lse.squeeze()
#                 + np.log(np.sum(np.exp(log_w - lse), axis=1)))


# # =============================================================================
# #  ALIGNMENT
# # =============================================================================
# def pecah_frame_per_kata(fitur, kata_list):
#     """Alignment naif berbasis panjang fonem."""
#     T, n = fitur.shape[0], len(kata_list)
#     if n == 0 or T == 0:
#         return {}
#     bobot   = np.array([max(len(kata_ke_fonem(k)), 1) for k in kata_list],
#                        dtype=float)
#     bobot  /= bobot.sum()
#     n_frame = (bobot * T).astype(int)
#     sisa    = T - n_frame.sum()
#     for i in range(sisa):
#         n_frame[i % n] += 1
#     hasil = {}
#     idx   = 0
#     for i, kata in enumerate(kata_list):
#         nf  = n_frame[i]
#         seg = fitur[idx: idx + nf]
#         if nf > 0:
#             hasil[kata] = (
#                 np.vstack([hasil[kata], seg]) if kata in hasil else seg
#             )
#         idx += nf
#     return hasil


# def pecah_frame_gmm(
#     fitur     : np.ndarray,
#     kata_list : List[str],
#     gmm_per_kata: Dict[str, "GaussianMixtureModel"],
# ) -> Dict[str, np.ndarray]:
#     """
#     [FIX-C] Monotonic forced alignment menggunakan GMM posterior.

#     Bug yang diperbaiki:
#     - Sebelumnya: bp[t, j] menyimpan indeks kata j, traceback tidak bisa
#       membedakan "tetap di j" vs "pindah dari j-1 ke j".
#     - Sekarang: bp[t, j] menyimpan ARAH (0 = tetap di j, 1 = pindah dari j-1),
#       sehingga traceback menghasilkan path yang dijamin monotonic.
#     """
#     T = fitur.shape[0]
#     n = len(kata_list)
#     if n == 0 or T == 0:
#         return {}
#     if not gmm_per_kata or not any(k in gmm_per_kata for k in kata_list):
#         return pecah_frame_per_kata(fitur, kata_list)

#     # Skor GMM per frame per kata: shape (T, n)
#     skor = np.full((T, n), 0.0, dtype=np.float64)
#     for j, kata in enumerate(kata_list):
#         if kata in gmm_per_kata:
#             skor[:, j] = gmm_per_kata[kata].log_prob_per_frame(fitur)

#     # DP monotonic forced alignment
#     INF = LOG_ZERO
#     dp  = np.full((T + 1, n + 1), INF)
#     # bp[t, j] = 0 (tetap di j) atau 1 (pindah dari j-1 ke j)
#     bp  = np.zeros((T + 1, n + 1), dtype=np.int8)
#     dp[0, 0] = 0.0

#     for t in range(1, T + 1):
#         for j in range(n):
#             # Opsi 1: frame t-1 juga ada di kata j (tetap)
#             score_stay   = dp[t - 1, j] + skor[t - 1, j] if dp[t - 1, j] > INF else INF
#             # Opsi 2: pindah dari kata j-1 ke kata j
#             score_advance = dp[t - 1, j - 1] + skor[t - 1, j] if (j > 0 and dp[t - 1, j - 1] > INF) else INF

#             if score_stay >= score_advance:
#                 dp[t, j] = score_stay
#                 bp[t, j] = 0   # tetap di j
#             else:
#                 dp[t, j] = score_advance
#                 bp[t, j] = 1   # pindah dari j-1

#     # Traceback — mengikuti backpointer arah
#     path = np.zeros(T, dtype=int)
#     j = n - 1
#     for t in range(T, 0, -1):
#         path[t - 1] = j
#         if bp[t, j] == 1:
#             j = j - 1  # frame ini adalah awal kata j, frame sebelumnya di j-1
#         # else: j tetap sama

#     # Kumpulkan frame per kata
#     hasil: Dict[str, np.ndarray] = {}
#     for kata in kata_list:
#         hasil[kata] = np.empty((0, fitur.shape[1]), dtype=np.float32)

#     for t, j in enumerate(path):
#         kata = kata_list[j]
#         hasil[kata] = np.vstack(
#             [hasil[kata], fitur[t: t + 1]]
#         ) if hasil[kata].shape[0] > 0 else fitur[t: t + 1]

#     # Fallback untuk kata yang tidak mendapat frame
#     naif = pecah_frame_per_kata(fitur, kata_list)
#     for kata in kata_list:
#         if hasil[kata].shape[0] == 0 and kata in naif:
#             hasil[kata] = naif[kata]

#     return hasil


# def kumpulkan_frame_per_kata(
#     cache_mfcc    : Dict[str, np.ndarray],
#     dict_transkrip: Dict[str, str],
#     gmm_per_kata  : Optional[Dict[str, "GaussianMixtureModel"]] = None,
# ) -> Dict[str, List[np.ndarray]]:
#     frame_per_kata: Dict[str, List[np.ndarray]] = defaultdict(list)
#     use_gmm = gmm_per_kata is not None and len(gmm_per_kata) > 0
#     print(f"  Mengumpulkan frame dari {len(cache_mfcc)} audio "
#           f"({'GMM' if use_gmm else 'naif'}) ...")
#     for audio_id, fitur in cache_mfcc.items():
#         label = dict_transkrip.get(audio_id, "")
#         if not label:
#             continue
#         kata_list = normalisasi(label).split()
#         pecahan   = (pecah_frame_gmm(fitur, kata_list, gmm_per_kata)
#                      if use_gmm
#                      else pecah_frame_per_kata(fitur, kata_list))
#         for kata, frame in pecahan.items():
#             if frame.shape[0] > 0:
#                 frame_per_kata[kata].append(frame)
#     print(f"  Kata unik: {len(frame_per_kata)}")
#     return frame_per_kata


# def latih_gmm_per_kata(
#     frame_per_kata, n_components=4, n_iter=100,
#     tol=1e-3, min_frame=10, verbose=True,
# ):
#     gmm_per_kata = {}
#     kata_list    = sorted(frame_per_kata.keys())
#     n_kata       = len(kata_list)
#     if verbose:
#         print(f"  Melatih GMM untuk {n_kata} kata ...")
#     t0 = time.time()
#     for i, kata in enumerate(kata_list):
#         X = np.vstack(frame_per_kata[kata])
#         if X.shape[0] < min_frame:
#             continue
#         gmm = GaussianMixtureModel(
#             n_components=min(n_components, X.shape[0]),
#             n_iter=n_iter, tol=tol, reg_covar=COV_REG,
#             random_state=hash(kata) % 100_000,
#         )
#         gmm.fit(X)
#         gmm_per_kata[kata] = gmm
#         if verbose and (i + 1) % 50 == 0:
#             print(f"    [{i+1}/{n_kata}] elapsed={time.time()-t0:.1f}s")
#     if verbose:
#         print(f"  GMM selesai: {len(gmm_per_kata)} kata dalam {time.time()-t0:.1f}s")
#     return gmm_per_kata


# # =============================================================================
# #  [FIX-B] TDNN DENGAN CTC LOSS (sequence-to-sequence)
# # =============================================================================
# class TDNNLayer(nn.Module):
#     def __init__(self, input_dim: int, output_dim: int, context: List[int],
#                  dropout: float = 0.0):
#         super().__init__()
#         self.context   = sorted(context)
#         self.input_dim = input_dim
#         n_ctx          = len(context)
#         self.linear    = nn.Linear(input_dim * n_ctx, output_dim)
#         self.bn        = nn.BatchNorm1d(output_dim)
#         self.drop      = nn.Dropout(dropout)

#     def forward(self, x: torch.Tensor) -> torch.Tensor:
#         B, T, D = x.shape
#         ctx_min = min(self.context)
#         ctx_max = max(self.context)
#         pad_l   = max(0, -ctx_min)
#         pad_r   = max(0,  ctx_max)
#         x_pad   = F.pad(x.permute(0, 2, 1), (pad_l, pad_r)).permute(0, 2, 1)
#         frames  = []
#         for c in self.context:
#             shift = c - ctx_min
#             frames.append(x_pad[:, shift: shift + T, :])
#         spliced = torch.cat(frames, dim=-1)
#         out     = self.linear(spliced)
#         out     = self.bn(out.permute(0, 2, 1)).permute(0, 2, 1)
#         out     = F.relu(out)
#         return self.drop(out)


# class TDNNAcousticModel(nn.Module):
#     """
#     [FIX-B] Model TDNN untuk CTC: output adalah log-softmax per frame,
#     bukan rata-rata per segmen. Jumlah kelas = len(vocab) + 1 (blank CTC).
#     """
#     def __init__(self, input_dim: int, n_classes: int,
#                  hidden_dim: int = 512,
#                  contexts: List[List[int]] = None,
#                  dropout: float = 0.1):
#         super().__init__()
#         if contexts is None:
#             contexts = [[-1, 0, 1], [-1, 0, 1], [-3, 0, 3], [-3, 0, 3], [-6, 0, 6]]
#         layers = []
#         in_dim = input_dim
#         for ctx in contexts:
#             layers.append(TDNNLayer(in_dim, hidden_dim, ctx, dropout))
#             in_dim = hidden_dim
#         self.tdnn_layers = nn.ModuleList(layers)
#         # n_classes sudah termasuk blank (indeks 0)
#         self.classifier  = nn.Linear(hidden_dim, n_classes)

#     def forward(self, x: torch.Tensor) -> torch.Tensor:
#         """x: (B, T, D) → (B, T, n_classes) log-softmax per frame."""
#         for layer in self.tdnn_layers:
#             x = layer(x)
#         logits = self.classifier(x)
#         return F.log_softmax(logits, dim=-1)


# # =============================================================================
# #  [FIX-B & FIX-F] DATASET CTC + COLLATE
# # =============================================================================
# class CTCDataset(Dataset):
#     """
#     [FIX-B] Dataset untuk CTC: setiap sampel adalah pasangan
#     (sekuens_fitur, sekuens_label_kata).
#     """
#     def __init__(
#         self,
#         cache_mfcc    : Dict[str, np.ndarray],
#         dict_transkrip: Dict[str, str],
#         vocab         : List[str],
#     ):
#         self.w2i     = {w: i + 1 for i, w in enumerate(vocab)}  # 0 = blank CTC
#         self.samples = []
#         for audio_id, fitur in cache_mfcc.items():
#             label = dict_transkrip.get(audio_id, "")
#             if not label:
#                 continue
#             kata_list = normalisasi(label).split()
#             ids       = [self.w2i[k] for k in kata_list if k in self.w2i]
#             if not ids:
#                 continue
#             self.samples.append((
#                 torch.tensor(fitur.astype(np.float32)),
#                 torch.tensor(ids, dtype=torch.long),
#             ))

#     def __len__(self):
#         return len(self.samples)

#     def __getitem__(self, i):
#         return self.samples[i]


# def collate_ctc(batch):
#     """
#     [FIX-F] Gunakan pad_sequence dari PyTorch untuk padding yang efisien.
#     Return: (padded_feats, targets, input_lengths, target_lengths)
#     CTC loss membutuhkan input_lengths dan target_lengths.
#     """
#     feats, targets = zip(*batch)
#     input_lengths  = torch.tensor([f.shape[0] for f in feats], dtype=torch.long)
#     target_lengths = torch.tensor([t.shape[0] for t in targets], dtype=torch.long)
#     # pad_sequence mengharapkan list tensor, menghasilkan (T_max, B, D)
#     padded_feats = pad_sequence(feats, batch_first=True, padding_value=0.0)
#     # Gabungkan semua target untuk CTC (flat, bukan padded)
#     flat_targets = torch.cat(targets)
#     return padded_feats, flat_targets, input_lengths, target_lengths


# # =============================================================================
# #  [FIX-B & FIX-K] TRAINING TDNN-CTC
# # =============================================================================
# def latih_tdnn_ctc(
#     cache_mfcc    : Dict[str, np.ndarray],
#     dict_transkrip: Dict[str, str],
#     vocab         : List[str],
#     dir_save      : str,
#     epochs        : int   = TDNN_EPOCHS,
#     lr            : float = TDNN_LR,
#     batch_size    : int   = TDNN_BATCH_SIZE,
#     hidden_dim    : int   = TDNN_HIDDEN,
#     contexts      : List[List[int]] = None,
#     dropout       : float = TDNN_DROPOUT,
#     patience      : int   = TDNN_PATIENCE,
#     val_ratio     : float = VAL_RATIO,
#     verbose       : bool  = True,
# ) -> TDNNAcousticModel:
#     if contexts is None:
#         contexts = TDNN_CONTEXTS

#     device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#     model_path = os.path.join(dir_save, "model_tdnn_ctc.pt")
#     n_classes  = len(vocab) + 1  # +1 untuk blank (indeks 0)
#     print(f"\n  Training TDNN-CTC pada device: {device}")
#     print(f"  Vocab: {len(vocab)} kata | n_classes (termasuk blank): {n_classes}")

#     model = TDNNAcousticModel(
#         input_dim  = FEAT_DIM,
#         n_classes  = n_classes,
#         hidden_dim = hidden_dim,
#         contexts   = contexts,
#         dropout    = dropout,
#     ).to(device)

#     # [FIX-K] Validasi dan load checkpoint jika sudah ada
#     if os.path.isfile(model_path):
#         try:
#             state = torch.load(model_path, map_location=device)
#             model.load_state_dict(state)
#             print(f"  [FIX-K] Checkpoint valid ditemukan, model dimuat: {model_path}")
#             return model
#         except (RuntimeError, Exception) as e:
#             print(f"  [FIX-K] Checkpoint korup ({e}), training ulang ...")
#             os.remove(model_path)

#     # Dataset dan split
#     full_dataset = CTCDataset(cache_mfcc, dict_transkrip, vocab)
#     n_total      = len(full_dataset)
#     n_val        = max(1, int(n_total * val_ratio))
#     n_train      = n_total - n_val

#     # [FIX-B] Shuffle sebelum split agar tidak bias per kata
#     indices      = list(range(n_total))
#     random.shuffle(indices)
#     train_idx    = indices[n_val:]
#     val_idx      = indices[:n_val]

#     from torch.utils.data import Subset
#     train_loader = DataLoader(
#         Subset(full_dataset, train_idx),
#         batch_size=batch_size, shuffle=True,
#         collate_fn=collate_ctc, num_workers=0,
#     )
#     val_loader = DataLoader(
#         Subset(full_dataset, val_idx),
#         batch_size=batch_size, shuffle=False,
#         collate_fn=collate_ctc, num_workers=0,
#     )
#     print(f"  Train: {n_train} | Val: {n_val}")

#     optimizer   = torch.optim.Adam(model.parameters(), lr=lr)
#     scheduler   = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
#     ctc_loss_fn = nn.CTCLoss(blank=0, reduction="mean", zero_infinity=True)

#     riwayat          = []
#     best_val_loss    = float("inf")
#     patience_counter = 0
#     t0               = time.time()

#     for epoch in range(1, epochs + 1):
#         # ── Train ──
#         model.train()
#         total_loss = 0.0
#         n_batch    = 0
#         for feats, targets, in_lens, tgt_lens in train_loader:
#             feats   = feats.to(device)
#             targets = targets.to(device)
#             in_lens = in_lens.to(device)

#             optimizer.zero_grad()
#             log_probs = model(feats)                          # (B, T, C)
#             # CTC loss membutuhkan (T, B, C)
#             log_probs_t = log_probs.permute(1, 0, 2)
#             loss = ctc_loss_fn(log_probs_t, targets, in_lens, tgt_lens)
#             if torch.isfinite(loss):
#                 loss.backward()
#                 nn.utils.clip_grad_norm_(model.parameters(), 5.0)
#                 optimizer.step()
#                 total_loss += loss.item()
#                 n_batch    += 1

#         # ── Validasi ──
#         model.eval()
#         val_loss   = 0.0
#         n_val_batch = 0
#         with torch.no_grad():
#             for feats, targets, in_lens, tgt_lens in val_loader:
#                 feats   = feats.to(device)
#                 targets = targets.to(device)
#                 in_lens = in_lens.to(device)
#                 lp      = model(feats).permute(1, 0, 2)
#                 v       = ctc_loss_fn(lp, targets, in_lens, tgt_lens)
#                 if torch.isfinite(v):
#                     val_loss    += v.item()
#                     n_val_batch += 1

#         avg_train = total_loss / max(n_batch, 1)
#         avg_val   = val_loss   / max(n_val_batch, 1)
#         scheduler.step()
#         riwayat.append({"Epoch": epoch,
#                          "Train Loss": round(avg_train, 6),
#                          "Val Loss": round(avg_val, 6)})

#         if verbose:
#             print(f"  Epoch {epoch:3d}/{epochs}  "
#                   f"train={avg_train:.4f}  val={avg_val:.4f}  "
#                   f"lr={scheduler.get_last_lr()[0]:.2e}  "
#                   f"elapsed={time.time()-t0:.0f}s")

#         # [FIX-K] Early stopping + simpan best model
#         if avg_val < best_val_loss:
#             best_val_loss    = avg_val
#             patience_counter = 0
#             torch.save(model.state_dict(), model_path)
#             if verbose:
#                 print(f"    → Best model disimpan (val={avg_val:.4f})")
#         else:
#             patience_counter += 1
#             if patience_counter >= patience:
#                 print(f"  Early stopping epoch {epoch} (patience={patience})")
#                 break

#     # [FIX-K] Eksplisit muat best model setelah training selesai
#     if os.path.isfile(model_path):
#         model.load_state_dict(torch.load(model_path, map_location=device))
#         print("  [FIX-K] Best model dimuat kembali setelah training.")

#     pd.DataFrame(riwayat).to_csv(
#         os.path.join(dir_save, "HasilTrainingTDNN.csv"), index=False
#     )
#     print(f"  Model TDNN-CTC disimpan → {model_path}")
#     return model


# # =============================================================================
# #  [FIX-A & FIX-D] DECODER TDNN-CTC DENGAN GREEDY/BEAM SEARCH YANG BENAR
# # =============================================================================
# class TDNNCTCDecoder:
#     """
#     [FIX-A] Decoder tidak menerima kata referensi — murni beroperasi pada
#     distribusi posterior akustik per frame.

#     [FIX-D] Beam search berbasis CTC: ekspansi per frame, bukan per kata.
#     Rescore dengan n-gram language model.
#     """
#     def __init__(
#         self,
#         model      : TDNNAcousticModel,
#         vocab      : List[str],
#         ngram_model: "NGramModel",
#         device     : torch.device = None,
#         beam_width : int = BEAM_WIDTH,
#     ):
#         self.model      = model
#         # vocab[0] = blank (internal), vocab_words[i] = kata ke-i (indeks i+1)
#         self.vocab      = vocab          # list kata (tanpa blank)
#         self.ngram      = ngram_model
#         self.device     = device or torch.device("cpu")
#         self.beam_width = beam_width
#         self.blank_id   = 0
#         self.model.to(self.device)
#         self.model.eval()

#     def _get_log_probs(self, fitur: np.ndarray) -> np.ndarray:
#         """Forward pass TDNN → (T, n_classes) numpy."""
#         with torch.no_grad():
#             t = torch.tensor(fitur, dtype=torch.float32).unsqueeze(0).to(self.device)
#             return self.model(t)[0].cpu().numpy()   # (T, n_classes)

#     def greedy_decode(self, fitur: np.ndarray) -> List[str]:
#         """Greedy CTC decode: argmax per frame + collapse blank/repeat."""
#         log_probs = self._get_log_probs(fitur)       # (T, C)
#         ids       = np.argmax(log_probs, axis=1)     # (T,)
#         # Collapse: hapus blank dan repeat berurutan
#         result = []
#         prev   = self.blank_id
#         for cid in ids:
#             if cid != self.blank_id and cid != prev:
#                 result.append(self.vocab[cid - 1])   # offset blank
#             prev = cid
#         return result

#     def beam_search_decode(self, fitur: np.ndarray) -> List[str]:
#         """
#         [FIX-D] CTC Beam Search yang benar:
#         Setiap state beam adalah (prefix_words, last_token_id).
#         Ekspansi dilakukan per frame atas distribusi posterior.
#         """
#         log_probs = self._get_log_probs(fitur)   # (T, C)
#         T, C      = log_probs.shape

#         # Beam state: dict { (kata_tuple, last_id) : log_prob }
#         # Inisialisasi dengan hipotesis kosong
#         beam: Dict[Tuple, float] = {((), self.blank_id): 0.0}

#         for t in range(T):
#             lp_t  = log_probs[t]           # (C,)
#             new_beam: Dict[Tuple, float] = defaultdict(lambda: LOG_ZERO)

#             for (prefix, last_id), score in beam.items():
#                 for cid in range(C):
#                     new_score = score + float(lp_t[cid])

#                     if cid == self.blank_id:
#                         # Blank: prefix tidak berubah, last_id → blank
#                         key = (prefix, self.blank_id)
#                         new_beam[key] = np.logaddexp(new_beam[key], new_score)

#                     elif cid == last_id:
#                         # Repeat token: hanya diterima jika sebelumnya blank
#                         if last_id == self.blank_id:
#                             new_prefix = prefix + (cid,)
#                             key        = (new_prefix, cid)
#                             new_beam[key] = np.logaddexp(new_beam[key], new_score)
#                         else:
#                             # Repeat non-blank: dianggap lanjutan token sama
#                             key = (prefix, cid)
#                             new_beam[key] = np.logaddexp(new_beam[key], new_score)

#                     else:
#                         # Token baru berbeda
#                         new_prefix = prefix + (cid,)
#                         key        = (new_prefix, cid)
#                         new_beam[key] = np.logaddexp(new_beam[key], new_score)

#             # Pertahankan beam_width hipotesis terbaik
#             sorted_beam = sorted(new_beam.items(), key=lambda x: x[1], reverse=True)
#             beam = dict(sorted_beam[: self.beam_width])

#         # Kumpulkan kandidat unik berdasarkan prefix kata
#         kandidat_dict: Dict[tuple, float] = {}
#         for (prefix, _), score in beam.items():
#             kata_prefix = tuple(
#                 self.vocab[cid - 1] for cid in prefix if cid != self.blank_id
#             )
#             if kata_prefix not in kandidat_dict or score > kandidat_dict[kata_prefix]:
#                 kandidat_dict[kata_prefix] = score

#         if not kandidat_dict:
#             return []

#         # Rescore dengan n-gram LM
#         kandidat_list = [list(k) for k in kandidat_dict.keys()]
#         return self.ngram.rescore(kandidat_list)

#     def dekode_audio(self, fitur: np.ndarray) -> List[str]:
#         """
#         [FIX-A] Tidak ada kata_ref_list — decoder sepenuhnya bebas.
#         Menggunakan beam search CTC jika beam_width > 1, greedy jika = 1.
#         """
#         if fitur.shape[0] == 0:
#             return []
#         if self.beam_width == 1:
#             return self.greedy_decode(fitur)
#         return self.beam_search_decode(fitur)


# # =============================================================================
# #  N-GRAM LANGUAGE MODEL (tidak berubah, sudah benar)
# # =============================================================================
# class NGramModel:
#     def __init__(self, n: int = 3, discount: float = 0.75):
#         self.n             = n
#         self.discount      = discount
#         self.ngram_count   = defaultdict(int)
#         self.context_count = defaultdict(int)
#         self.vocab         = set()
#         self.N_total       = 0
#         self._ctx_diversity: Dict[tuple, int] = {}

#     def _tokenize(self, kalimat: str) -> List[str]:
#         tokens = normalisasi(kalimat).split()
#         return ["<s>"] * (self.n - 1) + tokens + ["</s>"]

#     def fit(self, korpus: List[str]) -> "NGramModel":
#         ctx_words: Dict[tuple, set] = defaultdict(set)
#         for kalimat in korpus:
#             tokens = self._tokenize(kalimat)
#             self.vocab.update(tokens)
#             self.N_total += len(tokens)
#             for i in range(len(tokens) - self.n + 1):
#                 ngram = tuple(tokens[i: i + self.n])
#                 ctx   = ngram[:-1]
#                 self.ngram_count[ngram]  += 1
#                 self.context_count[ctx]  += 1
#                 ctx_words[ctx].add(ngram[-1])
#         self._ctx_diversity = {c: len(w) for c, w in ctx_words.items()}
#         return self

#     def log_prob(self, kata: str, context: tuple) -> float:
#         ngram = context + (kata,)
#         C_ng  = self.ngram_count.get(ngram, 0)
#         C_ctx = self.context_count.get(context, 0)
#         if C_ctx == 0:
#             C_uni = self.ngram_count.get((kata,), 0)
#             return math.log(
#                 (C_uni + 1) / (self.N_total + len(self.vocab)) + LOG_EPS
#             )
#         lam   = self.discount * self._ctx_diversity.get(context, 0) / C_ctx
#         C_uni = self.ngram_count.get((kata,), 0)
#         p_back = (C_uni + 1) / (self.N_total + len(self.vocab))
#         prob   = max(C_ng - self.discount, 0) / C_ctx + lam * p_back
#         return math.log(prob + LOG_EPS)

#     def score_sequence(self, kata_list: List[str]) -> float:
#         tokens = ["<s>"] * (self.n - 1) + kata_list + ["</s>"]
#         log_p  = 0.0
#         for i in range(self.n - 1, len(tokens)):
#             log_p += self.log_prob(
#                 tokens[i], tuple(tokens[i - self.n + 1: i])
#             )
#         return log_p

#     def rescore(self, kandidat: List[List[str]]) -> List[str]:
#         if not kandidat:
#             return []
#         return max(kandidat, key=self.score_sequence)


# # =============================================================================
# #  COMPUTE WER (traceback benar, dari versi sebelumnya)
# # =============================================================================
# def compute_wer(referensi: List[str], hipotesis: List[str]) -> Dict:
#     R, H = len(referensi), len(hipotesis)
#     dp   = [[0] * (H + 1) for _ in range(R + 1)]
#     ops  = [[""] * (H + 1) for _ in range(R + 1)]

#     for i in range(R + 1): dp[i][0] = i; ops[i][0] = "D" * i
#     for j in range(H + 1): dp[0][j] = j; ops[0][j] = "I" * j

#     for i in range(1, R + 1):
#         for j in range(1, H + 1):
#             if referensi[i - 1] == hipotesis[j - 1]:
#                 dp[i][j]  = dp[i - 1][j - 1]
#                 ops[i][j] = "M"
#             else:
#                 sub  = dp[i - 1][j - 1] + 1
#                 dlt  = dp[i - 1][j]     + 1
#                 ins  = dp[i][j - 1]     + 1
#                 best = min(sub, dlt, ins)
#                 dp[i][j]  = best
#                 if best == sub:   ops[i][j] = "S"
#                 elif best == dlt: ops[i][j] = "D"
#                 else:             ops[i][j] = "I"

#     S = D = I = 0
#     i, j = R, H
#     while i > 0 or j > 0:
#         op = ops[i][j]
#         if op == "M":
#             i -= 1; j -= 1
#         elif op == "S":
#             S += 1; i -= 1; j -= 1
#         elif op == "D":
#             D += 1; i -= 1
#         elif op == "I":
#             I += 1; j -= 1
#         else:
#             if i > 0: D += 1; i -= 1
#             else:     I += 1; j -= 1

#     N   = R
#     wer = (S + D + I) / N if N > 0 else 0.0
#     return {"wer_persen": round(wer * 100, 2), "S": S, "D": D, "I": I, "N": N}


# # =============================================================================
# #  UTILITY
# # =============================================================================
# def build_mfcc_cache(
#     sampled_root  : str,
#     dict_transkrip: Dict[str, str],
# ) -> Tuple[Dict[str, np.ndarray], List[dict]]:
#     wav_files  = glob.glob(os.path.join(sampled_root, "**/*.wav"),  recursive=True)
#     wav_files += glob.glob(os.path.join(sampled_root, "**/*.flac"), recursive=True)
#     cache_mfcc = {}
#     meta_list  = []
#     n_total    = len(wav_files)
#     print(f"  Mengekstrak MFCC dari {n_total} file ...")
#     t0 = time.time()
#     for idx, wav_path in enumerate(wav_files, 1):
#         audio_id = os.path.splitext(os.path.basename(wav_path))[0]
#         label    = dict_transkrip.get(audio_id, "")
#         if not label:
#             continue
#         # [FIX-J] Hanya I/O error yang di-catch, bug programmer dibiarkan naik
#         try:
#             fitur = extract_features(wav_path, cmvn=True)
#         except (FileNotFoundError, ValueError, OSError) as e:
#             print(f"    [SKIP I/O] {os.path.basename(wav_path)}: {e}")
#             continue
#         cache_mfcc[audio_id] = fitur
#         meta_list.append({
#             "Id": audio_id, "Label": label,
#             "Jumlah Frame": fitur.shape[0],
#             "Dimensi Fitur": fitur.shape[1],
#         })
#         if idx % 100 == 0:
#             ela = time.time() - t0
#             eta = ela / idx * (n_total - idx)
#             print(f"    [{idx}/{n_total}] elapsed={ela:.0f}s ETA={eta:.0f}s")
#     print(f"  Selesai: {len(cache_mfcc)} audio dalam {time.time()-t0:.1f}s")
#     return cache_mfcc, meta_list


# def dekode_semua_audio(
#     cache_mfcc    : Dict[str, np.ndarray],
#     dict_transkrip: Dict[str, str],
#     decoder       : "TDNNCTCDecoder",
#     verbose       : bool = True,
# ) -> List[dict]:
#     """
#     [FIX-A] Tidak ada kata referensi yang diteruskan ke decoder.
#     WER dihitung SETELAH mendapatkan hipotesis bebas.
#     """
#     hasil_wer = []
#     n_total   = len(cache_mfcc)
#     t0        = time.time()
#     if verbose:
#         print(f"  Dekode untuk {n_total} audio ...")
#     for idx, (audio_id, fitur) in enumerate(cache_mfcc.items(), 1):
#         label = dict_transkrip.get(audio_id, "")
#         if not label:
#             continue
#         ref_kata = normalisasi(label).split()
#         if not ref_kata:
#             continue
#         # [FIX-J] Hanya runtime error akustik yang di-catch
#         try:
#             hyp_kata = decoder.dekode_audio(fitur)
#         except (RuntimeError, MemoryError) as e:
#             if verbose:
#                 print(f"    [ERROR dekode] {audio_id}: {e}")
#             continue
#         wer_hasil = compute_wer(ref_kata, hyp_kata)
#         hasil_wer.append({
#             "Id": audio_id,
#             "Referensi": " ".join(ref_kata),
#             "Hipotesis": " ".join(hyp_kata),
#             "WER (%)": wer_hasil["wer_persen"],
#             "S": wer_hasil["S"], "D": wer_hasil["D"],
#             "I": wer_hasil["I"], "N": wer_hasil["N"],
#         })
#         if verbose and idx % 100 == 0:
#             print(f"    [{idx}/{n_total}] elapsed={time.time()-t0:.1f}s")
#     if verbose:
#         print(f"  Selesai: {len(hasil_wer)} audio dalam {time.time()-t0:.1f}s")
#     return hasil_wer


# def ringkasan_wer(df_wer: pd.DataFrame, label: str = "") -> Dict:
#     if df_wer.empty:
#         print("  [WARNING] Tidak ada data WER")
#         return {}
#     total_S = int(df_wer["S"].sum())
#     total_D = int(df_wer["D"].sum())
#     total_I = int(df_wer["I"].sum())
#     total_N = int(df_wer["N"].sum())
#     wer_corpus = (
#         (total_S + total_D + total_I) / total_N * 100 if total_N > 0 else 100.0
#     )
#     print(f"\n  Ringkasan WER {label}")
#     print(f"  Jumlah audio  : {len(df_wer)}")
#     print(f"  WER Corpus    : {wer_corpus:.2f}%")
#     print(f"  WER Mean      : {df_wer['WER (%)'].mean():.2f}%")
#     print(f"  WER Median    : {df_wer['WER (%)'].median():.2f}%")
#     print(f"  WER Std       : {df_wer['WER (%)'].std():.2f}%")
#     print(f"  WER Min       : {df_wer['WER (%)'].min():.2f}%")
#     print(f"  WER Max       : {df_wer['WER (%)'].max():.2f}%")
#     print(f"  Total S/D/I/N : {total_S}/{total_D}/{total_I}/{total_N}")
#     return {
#         "n_audio": len(df_wer), "wer_corpus": round(wer_corpus, 2),
#         "wer_mean": round(df_wer["WER (%)"].mean(), 2),
#         "wer_median": round(df_wer["WER (%)"].median(), 2),
#         "wer_std": round(df_wer["WER (%)"].std(), 2),
#         "wer_min": round(df_wer["WER (%)"].min(), 2),
#         "wer_max": round(df_wer["WER (%)"].max(), 2),
#         "total_S": total_S, "total_D": total_D,
#         "total_I": total_I, "total_N": total_N,
#     }


# # =============================================================================
# #  MAIN
# # =============================================================================
# def main():
#     os.makedirs(DIR_SAVE, exist_ok=True)
#     t_start = time.time()

#     # ── 1. Baca data ──────────────────────────────────────────────────────────
#     print("\n[1] Membaca transkrip ...")
#     dict_transkrip  = baca_transkrip_csv(CSV_PATH)
#     semua_transkrip = list(dict_transkrip.values())

#     # ── 2. Lexicon ────────────────────────────────────────────────────────────
#     print("\n[2] Membangun lexicon ...")
#     lexicon = buat_lexicon(semua_transkrip)
#     pd.DataFrame([
#         {"Kata": k, "Fonem": " ".join(v)} for k, v in lexicon.items()
#     ]).to_csv(f"{DIR_SAVE}/HasilLexicon.csv", index=False)

#     # ── 3. Ekstraksi MFCC + CMVN ─────────────────────────────────────────────
#     print("\n[3] Ekstraksi MFCC + CMVN ...")
#     cache_mfcc, meta_list = build_mfcc_cache(SAMPLED_ROOT, dict_transkrip)
#     pd.DataFrame(meta_list).to_csv(f"{DIR_SAVE}/HasilMFCC.csv", index=False)

#     if not cache_mfcc:
#         print("  [ERROR] Tidak ada audio berhasil di-load.")
#         return

#     # ── 4. GMM Aligner ────────────────────────────────────────────────────────
#     print("\n[4a] GMM aligner — alignment naif (iterasi-1) ...")
#     frame_per_kata_naif = kumpulkan_frame_per_kata(
#         cache_mfcc, dict_transkrip, gmm_per_kata=None
#     )
#     gmm_per_kata = latih_gmm_per_kata(
#         frame_per_kata_naif,
#         n_components=N_COMPONENTS_GMM,
#         n_iter=N_ITER_GMM, tol=TOL_GMM,
#     )
#     pd.DataFrame([
#         {"Kata": k,
#          "Jumlah Frame": np.vstack(frame_per_kata_naif[k]).shape[0],
#          "K": g.n_components, "Iterasi": g.n_iter_done_,
#          "Konvergen": g.converged_,
#          "LL Akhir": round(g.log_likelihood_history_[-1], 4)}
#         for k, g in gmm_per_kata.items()
#     ]).to_csv(f"{DIR_SAVE}/HasilGMMAligner.csv", index=False)

#     # Re-alignment dengan GMM posterior
#     print("\n[4b] Re-alignment dengan GMM posterior [FIX-C] ...")
#     frame_per_kata = kumpulkan_frame_per_kata(
#         cache_mfcc, dict_transkrip, gmm_per_kata=gmm_per_kata
#     )

#     # ── 5. Language model ─────────────────────────────────────────────────────
#     print("\n[5] Melatih language model n-gram ...")
#     ngram_model = NGramModel(n=N_GRAM, discount=0.75)
#     ngram_model.fit(semua_transkrip)

#     # ── 6. Training TDNN-CTC ─────────────────────────────────────────────────
#     print("\n[6] Training TDNN-CTC acoustic model [FIX-B] ...")
#     vocab      = sorted({
#         kata
#         for label in dict_transkrip.values()
#         for kata in normalisasi(label).split()
#     })
#     tdnn_model = latih_tdnn_ctc(
#         cache_mfcc    = cache_mfcc,
#         dict_transkrip= dict_transkrip,
#         vocab         = vocab,
#         dir_save      = DIR_SAVE,
#         epochs        = TDNN_EPOCHS,
#         lr            = TDNN_LR,
#         batch_size    = TDNN_BATCH_SIZE,
#         hidden_dim    = TDNN_HIDDEN,
#         contexts      = TDNN_CONTEXTS,
#         dropout       = TDNN_DROPOUT,
#         patience      = TDNN_PATIENCE,
#         val_ratio     = VAL_RATIO,
#     )

#     # ── 7. Decoding ───────────────────────────────────────────────────────────
#     print("\n[7] Dekode dengan CTC beam search [FIX-A & FIX-D] ...")
#     device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#     decoder = TDNNCTCDecoder(
#         tdnn_model, vocab, ngram_model,
#         device=device, beam_width=BEAM_WIDTH,
#     )

#     hasil_wer = dekode_semua_audio(cache_mfcc, dict_transkrip, decoder)
#     df_wer    = pd.DataFrame(hasil_wer)
#     df_wer.to_csv(f"{DIR_SAVE}/HasilWER_HybridGMM_TDNN.csv", index=False)

#     stat = ringkasan_wer(df_wer, label="Hybrid GMM-TDNN v2")
#     pd.DataFrame([stat]).to_csv(
#         f"{DIR_SAVE}/HasilRingkasanWER_Hybrid.csv", index=False
#     )

#     print(f"\n  Selesai dalam {(time.time()-t_start)/60:.1f} menit")
#     print(f"  Output: {DIR_SAVE}")


# if __name__ == "__main__":
#     main()

In [ ]:
# =============================================================================
#  TDNN-HMM ASR – Bahasa Jawa  [v3 – Mengikuti Paper Ing et al. 2022]
#
#  PERBAIKAN FUNDAMENTAL DARI v2:
#
#  [CORE-1] Unit FONEM bukan kata:
#           TDNN output → posterior fonem per frame (bukan CTC word-level)
#           Output space: ~30 fonem vs 500+ kata → training jauh lebih efektif
#           Data per kelas: 50-100x lebih banyak → akurasi TDNN meningkat drastis
#
#  [CORE-2] Loss function: Cross-entropy per frame (bukan CTC):
#           Setiap frame audio diberi label fonem dari hasil GMM alignment
#           TDNN belajar p(fonem | frame) → probabilitas akustik yang kuat
#
#  [CORE-3] GMM per FONEM (bukan per kata):
#           Fonem muncul di seluruh vocab → ratusan ribu frame per fonem
#           Alignment akurat karena GMM dilatih dengan data yang cukup
#           Mengikuti paper: monophone GMM sebagai aligner untuk TDNN
#
#  [CORE-4] Decoding: TDNN-HMM Viterbi (bukan CTC beam search):
#           posterior fonem → HMM N-states per fonem → Viterbi → fonem seq
#           → lexicon lookup → kata sequence → n-gram LM rescore
#           Identik dengan arsitektur paper yang menghasilkan WER 0.97%
#
#  [CORE-5] Split berdasarkan speaker ID (bukan random):
#           Mencegah speaker leakage antara train dan test
#           Mengikuti EXP1 paper: speaker tertentu di-hold-out untuk evaluasi
#
#  [RETAIN dari v2]: vectorized GMM, CMVN, delta numpy, checkpoint, LM
# =============================================================================

SAMPLED_ROOT = r"D:\doc\2026\FINAL_CODE\data\sampled_small"

CSV_PATH = (
    r"D:\doc\2026\FINAL_CODE\src\codeStat\preModelling"
    r"\samplingStratified\out_samplingStratified\dataSampledSmall.csv"
)

DIR_SAVE = (
    r"D:\doc\2026\FINAL_CODE\src\codeStat\modelling"
    r"\voiceModelling\srcVoiceModelling\hybrid\src\TDNN-HMM\out"
)

# Kolom speaker di CSV (opsional — jika ada, digunakan untuk speaker split)
# Jika tidak ada kolom speaker, set ke None → fallback ke random split
COL_SPEAKER = "speaker_id"  # sesuaikan dengan nama kolom di CSV Anda

# ── Hyperparameter GMM ────────────────────────────────────────────────────────
N_COMPONENTS_GMM = 16   # komponen GMM per fonem
N_ITER_GMM       = 100
TOL_GMM          = 1e-4
MIN_FRAME_GMM    = 10   # minimum frame untuk latih GMM suatu fonem

# ── Hyperparameter TDNN ───────────────────────────────────────────────────────
TDNN_HIDDEN      = 512
TDNN_DROPOUT     = 0.1
TDNN_EPOCHS      = 50
TDNN_LR          = 1e-3
TDNN_BATCH_SIZE  = 256   # [CORE-1] fonem-level: lebih banyak sample per epoch
TDNN_CONTEXTS    = [     # [SAMA seperti paper] time strides simetris
    [-1, 0, 1],
    [-1, 0, 1],
    [-3, 0, 3],
    [-3, 0, 3],
    [-6, 0, 6],
]
TDNN_PATIENCE    = 7

# ── HMM ──────────────────────────────────────────────────────────────────────
N_HMM_STATES     = 3    # [MENGIKUTI PAPER] diuji 3, 4, 5 state per fonem
                         # Ganti ke 4 untuk eksperimen terbaik paper

# ── Language model ────────────────────────────────────────────────────────────
N_GRAM           = 3
BEAM_WIDTH       = 10   # lebar beam untuk Viterbi+LM rescore
VAL_RATIO        = 0.2
SPEAKER_SPLIT    = True  # [CORE-5] True = split per speaker, False = random

# ── Konstanta numerik ─────────────────────────────────────────────────────────
LOG_ZERO  = -1e15
LOG_EPS   = 1e-300
COV_REG   = 1e-6
FEAT_EPS  = 1e-8

# =============================================================================
#  IMPORT
# =============================================================================
import os, re, glob, math, time, random, warnings
from functools import lru_cache
from collections import defaultdict
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader, Subset
    from torch.nn.utils.rnn import pad_sequence
    _TORCH = True
except ImportError:
    raise ImportError("PyTorch diperlukan: pip install torch")

try:
    import librosa
    _LIBROSA = True
except ImportError:
    _LIBROSA = False

try:
    from scipy.io import wavfile as _wavfile
    _SCIPY = True
except ImportError:
    _SCIPY = False


# =============================================================================
#  [CORE-1] SET FONEM BAHASA JAWA
#  Mengikuti pendekatan paper: fonem eksplisit, bukan karakter langsung
#  PS27-style: semua monofonem, tanpa glottal stop (akurasi rendah)
# =============================================================================
# Fonem Bahasa Jawa Latin (28 fonem + SIL)
FONEM_JAWA: List[str] = [
    # Vokal (7)
    "a", "i", "u", "e", "o", "eu", "ae",
    # Stop (8)
    "p", "b", "t", "d", "k", "g", "dh", "th",
    # Nasal (4)
    "m", "n", "ng", "ny",
    # Frikatif (3)
    "s", "h", "f",
    # Afrikat (2)
    "j", "c",
    # Likuida (2)
    "l", "r",
    # Glide (2)
    "w", "y",
    # Senyap
    "SIL",
]
FONEM2ID: Dict[str, int] = {f: i for i, f in enumerate(FONEM_JAWA)}
N_FONEM: int = len(FONEM_JAWA)   # 29 (termasuk SIL)
SIL_ID: int  = FONEM2ID["SIL"]


# =============================================================================
#  G2P (Grapheme-to-Phoneme) Bahasa Jawa
#  [CORE-1] Mengubah kata → sekuens fonem menggunakan aturan Jawa Latin
# =============================================================================
_G2P_DIGRAF: Dict[str, str] = {
    "dh": "dh", "th": "th", "ng": "ng", "ny": "ny",
}

@lru_cache(maxsize=8192)
def kata_ke_fonem(kata: str) -> List[str]:
    """
    Konversi kata Bahasa Jawa → list fonem.
    Prioritaskan digraf (dh, th, ng, ny) sebelum monofonem.
    """
    fonem: List[str] = []
    i = 0
    while i < len(kata):
        # Cek digraf (2 karakter)
        if i + 1 < len(kata) and kata[i:i+2] in _G2P_DIGRAF:
            fonem.append(_G2P_DIGRAF[kata[i:i+2]])
            i += 2
        elif kata[i] in FONEM2ID:
            fonem.append(kata[i])
            i += 1
        else:
            # Karakter tidak dikenal → skip
            i += 1
    return fonem if fonem else ["SIL"]

def buat_lexicon(transkrip_list: List[str]) -> Dict[str, List[str]]:
    """Buat kamus kata → fonem dari seluruh transkripsi."""
    lex: Dict[str, List[str]] = {}
    for kalimat in transkrip_list:
        for kata in normalisasi(kalimat).split():
            if kata not in lex:
                lex[kata] = kata_ke_fonem(kata)
    return lex


# =============================================================================
#  NORMALISASI
# =============================================================================
@lru_cache(maxsize=4096)
def normalisasi(teks: str) -> str:
    teks = teks.lower().strip()
    teks = re.sub(r"[^a-z\s]", "", teks)
    return re.sub(r"\s+", " ", teks).strip()


# =============================================================================
#  BACA CSV
# =============================================================================
def baca_transkrip_csv(
    csv_path: str,
    col_speaker: Optional[str] = COL_SPEAKER,
) -> Tuple[Dict[str, str], Dict[str, str]]:
    """
    Return:
        dict_transkrip: {audio_id: transkrip}
        dict_speaker:   {audio_id: speaker_id}  (kosong jika tidak ada kolom)
    """
    if not os.path.isfile(csv_path):
        raise FileNotFoundError(f"CSV tidak ditemukan: {csv_path}")

    df = None
    for enc in ("utf-8", "utf-8-sig", "latin-1", "cp1252"):
        try:
            df = pd.read_csv(csv_path, encoding=enc)
            break
        except (UnicodeDecodeError, pd.errors.ParserError):
            continue
    if df is None:
        raise ValueError(f"Gagal membaca CSV: {csv_path}")

    df.columns = [c.strip() for c in df.columns]

    col_path_cands = ["Path_Data", "path_data", "path", "audio_path",
                      "filename", "file", "wav_path"]
    col_text_cands = ["sentences", "sentence", "text", "transcription",
                      "label", "transcript"]

    col_path = next((c for c in col_path_cands if c in df.columns), None)
    col_text = next((c for c in col_text_cands if c in df.columns), None)
    if col_path is None:
        raise ValueError(f"Kolom path tidak ditemukan. Kolom: {list(df.columns)}")
    if col_text is None:
        raise ValueError(f"Kolom transkrip tidak ditemukan. Kolom: {list(df.columns)}")

    dict_transkrip: Dict[str, str] = {}
    dict_speaker: Dict[str, str]   = {}
    has_speaker = col_speaker and col_speaker in df.columns

    for _, row in df.iterrows():
        pv = str(row[col_path]).strip()
        tv = str(row[col_text]).strip()
        if pv in ("nan", "") or tv in ("nan", ""):
            continue
        audio_id = os.path.splitext(os.path.basename(pv))[0]
        dict_transkrip[audio_id] = tv.lower()
        if has_speaker:
            dict_speaker[audio_id] = str(row[col_speaker]).strip()

    print(f"  CSV dibaca: {len(dict_transkrip)} entri, "
          f"{'dengan' if has_speaker else 'tanpa'} kolom speaker")
    return dict_transkrip, dict_speaker


# =============================================================================
#  [CORE-5] SPLIT BERDASARKAN SPEAKER
#  Mengikuti EXP1 paper: hold-out speaker tertentu untuk test set
# =============================================================================
def split_by_speaker(
    audio_ids: List[str],
    dict_speaker: Dict[str, str],
    val_ratio: float = VAL_RATIO,
) -> Tuple[List[str], List[str]]:
    """
    Bagi audio_ids ke train/val berdasarkan speaker.
    Speaker yang sama tidak akan muncul di kedua set.
    """
    if not dict_speaker:
        # Fallback: random split
        random.shuffle(audio_ids)
        n_val = max(1, int(len(audio_ids) * val_ratio))
        return audio_ids[n_val:], audio_ids[:n_val]

    # Kelompokkan audio per speaker
    spk2ids: Dict[str, List[str]] = defaultdict(list)
    for aid in audio_ids:
        spk = dict_speaker.get(aid, "unknown")
        spk2ids[spk].append(aid)

    speakers = list(spk2ids.keys())
    random.shuffle(speakers)
    n_val_spk = max(1, int(len(speakers) * val_ratio))

    val_speakers = set(speakers[:n_val_spk])
    train_ids, val_ids = [], []
    for spk, ids in spk2ids.items():
        if spk in val_speakers:
            val_ids.extend(ids)
        else:
            train_ids.extend(ids)

    print(f"  Speaker split: train={len(train_ids)} audio ({len(speakers)-n_val_spk} spk) "
          f"| val={len(val_ids)} audio ({n_val_spk} spk)")
    return train_ids, val_ids


# =============================================================================
#  EKSTRAKSI FITUR MFCC + DELTA + CMVN
#  [SAMA dengan paper]: MFCC 13 + delta + delta2 = 39 dim, 10ms/25ms
# =============================================================================
_SR         = 16000
_N_MFCC     = 13
_N_FFT      = 512
_HOP_LENGTH = 160   # 10ms
_WIN_LENGTH = 400   # 25ms
_N_MELS     = 26
_PRE_EMPH   = 0.97
_FMIN       = 0.0
_FMAX       = 4000.0
_DELTA_W    = 2
FEAT_DIM    = _N_MFCC * 3   # 39

def apply_cmvn(features: np.ndarray) -> np.ndarray:
    mean = features.mean(axis=0, keepdims=True)
    std  = features.std(axis=0, keepdims=True) + FEAT_EPS
    return (features - mean) / std

def _delta_numpy(feat: np.ndarray, w: int = _DELTA_W) -> np.ndarray:
    T = feat.shape[0]
    denom  = 2.0 * sum(n * n for n in range(1, w + 1))
    padded = np.pad(feat, ((w, w), (0, 0)), mode="edge")
    delta  = np.zeros_like(feat)
    for n in range(1, w + 1):
        delta += n * (padded[w+n:w+n+T] - padded[w-n:w-n+T])
    return delta / denom

def extract_features(wav_path: str, cmvn: bool = True) -> np.ndarray:
    if not os.path.isfile(wav_path):
        raise FileNotFoundError(f"File tidak ditemukan: {wav_path}")
    if _LIBROSA:
        feat = _mfcc_librosa(wav_path)
    elif _SCIPY:
        feat = _mfcc_scipy(wav_path)
    else:
        raise ImportError("Install librosa atau scipy")
    if cmvn:
        feat = apply_cmvn(feat)
    return feat

def _mfcc_librosa(wav_path: str) -> np.ndarray:
    y, _ = librosa.load(wav_path, sr=_SR, mono=True)
    if len(y) == 0:
        raise ValueError(f"Audio kosong: {wav_path}")
    y      = np.append(y[0], y[1:] - _PRE_EMPH * y[:-1])
    mfcc   = librosa.feature.mfcc(
        y=y, sr=_SR, n_mfcc=_N_MFCC,
        n_fft=_N_FFT, hop_length=_HOP_LENGTH,
        win_length=_WIN_LENGTH, n_mels=_N_MELS,
        fmin=_FMIN, fmax=_FMAX, window="hamming",
    )
    d1 = librosa.feature.delta(mfcc, width=_DELTA_W*2+1)
    d2 = librosa.feature.delta(mfcc, order=2, width=_DELTA_W*2+1)
    return np.vstack([mfcc, d1, d2]).T.astype(np.float32)

def _mfcc_scipy(wav_path: str) -> np.ndarray:
    sr, audio = _wavfile.read(wav_path)
    if audio.dtype == np.int16:
        audio = audio.astype(np.float32) / 32768.0
    elif audio.dtype == np.int32:
        audio = audio.astype(np.float32) / 2147483648.0
    else:
        audio = audio.astype(np.float32)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != _SR:
        n_new = int(len(audio) * _SR / sr)
        audio = np.interp(np.linspace(0, len(audio)-1, n_new),
                          np.arange(len(audio)), audio)
    y = np.append(audio[0], audio[1:] - _PRE_EMPH * audio[:-1])
    n_frames = max(1, 1 + (len(y) - _WIN_LENGTH) // _HOP_LENGTH)
    idx = np.arange(_WIN_LENGTH)[None,:] + _HOP_LENGTH * np.arange(n_frames)[:,None]
    idx = np.clip(idx, 0, len(y)-1)
    frames = y[idx].astype(np.float32) * np.hamming(_WIN_LENGTH).astype(np.float32)
    fft_mag = np.abs(np.fft.rfft(frames, n=_N_FFT))
    def hz2mel(f): return 1127.0 * np.log1p(f / 700.0)
    def mel2hz(m): return 700.0 * (np.exp(m / 1127.0) - 1.0)
    mel_pts = np.linspace(hz2mel(_FMIN), hz2mel(_FMAX), _N_MELS+2)
    hz_pts  = mel2hz(mel_pts)
    bins    = np.floor((_N_FFT+1) * hz_pts / _SR).astype(int)
    fb = np.zeros((_N_MELS, _N_FFT//2+1), dtype=np.float32)
    for m in range(1, _N_MELS+1):
        for k in range(bins[m-1], bins[m]):
            fb[m-1, k] = (k - bins[m-1]) / max(bins[m] - bins[m-1], 1)
        for k in range(bins[m], bins[m+1]):
            fb[m-1, k] = (bins[m+1] - k) / max(bins[m+1] - bins[m], 1)
    mel_e = np.maximum(fft_mag**2 @ fb.T, 1e-10)
    log_e = np.log10(mel_e)
    N = _N_MELS
    dct_m = np.cos(np.pi * np.arange(_N_MFCC)[:,None]
                   * (np.arange(N) + 0.5)[None,:] / N)
    mfcc = (log_e @ dct_m.T) * np.sqrt(2.0 / N)
    d1 = _delta_numpy(mfcc)
    d2 = _delta_numpy(d1)
    return np.concatenate([mfcc, d1, d2], axis=1).astype(np.float32)

def build_mfcc_cache(
    sampled_root  : str,
    dict_transkrip: Dict[str, str],
) -> Tuple[Dict[str, np.ndarray], List[dict]]:
    wav_files  = glob.glob(os.path.join(sampled_root, "**/*.wav"),  recursive=True)
    wav_files += glob.glob(os.path.join(sampled_root, "**/*.flac"), recursive=True)
    cache, meta = {}, []
    print(f"  Mengekstrak MFCC dari {len(wav_files)} file ...")
    t0 = time.time()
    for idx, wp in enumerate(wav_files, 1):
        aid = os.path.splitext(os.path.basename(wp))[0]
        if dict_transkrip.get(aid, "") == "":
            continue
        try:
            feat = extract_features(wp, cmvn=True)
        except (FileNotFoundError, ValueError, OSError) as e:
            print(f"    [SKIP] {os.path.basename(wp)}: {e}")
            continue
        cache[aid] = feat
        meta.append({"Id": aid, "Frames": feat.shape[0], "Dim": feat.shape[1]})
        if idx % 100 == 0:
            ela = time.time()-t0
            print(f"    [{idx}/{len(wav_files)}] {ela:.0f}s elapsed")
    print(f"  Selesai: {len(cache)} audio")
    return cache, meta


# =============================================================================
#  [CORE-3] GMM PER FONEM
#  Berbeda dari v2 (GMM per kata): fonem muncul di seluruh vocab
#  → data per GMM jauh lebih banyak → alignment akurat
# =============================================================================
class GaussianMixtureModel:
    def __init__(self, n_components=4, n_iter=100, tol=1e-3,
                 reg_covar=COV_REG, random_state=None):
        self.n_components = n_components
        self.n_iter       = n_iter
        self.tol          = tol
        self.reg_covar    = reg_covar
        self.random_state = random_state
        self.weights_     = None
        self.means_       = None
        self.covariances_ = None
        self.converged_   = False
        self.n_iter_done_ = 0
        self.log_likelihood_history_: List[float] = []

    def _init_params(self, X: np.ndarray):
        rng = np.random.default_rng(self.random_state)
        T, D = X.shape
        K    = self.n_components
        idx  = rng.choice(T, size=K, replace=False)
        self.means_       = X[idx].copy()
        self.weights_     = np.full(K, 1.0 / K)
        self.covariances_ = np.tile(np.var(X, axis=0) + self.reg_covar, (K, 1))

    def _log_gaussian_all(self, X: np.ndarray) -> np.ndarray:
        """Vectorized: X (T,D) → (T,K)."""
        mu  = self.means_
        var = self.covariances_
        log_det = np.sum(np.log(var), axis=1)
        diff    = X[:,None,:] - mu[None,:,:]
        maha    = np.sum(diff**2 / var[None,:,:], axis=2)
        log_norm = -0.5 * (X.shape[1] * np.log(2*np.pi) + log_det)
        return log_norm[None,:] - 0.5 * maha

    def _e_step(self, X):
        log_w = np.log(self.weights_ + LOG_EPS) + self._log_gaussian_all(X)
        lse   = np.max(log_w, axis=1, keepdims=True)
        log_sum = lse.squeeze() + np.log(np.sum(np.exp(log_w - lse), axis=1))
        return log_w - log_sum[:,None], log_sum

    def _m_step(self, X, log_resp):
        resp = np.exp(log_resp)
        Nk   = resp.sum(axis=0) + LOG_EPS
        self.weights_     = Nk / X.shape[0]
        self.means_       = (resp.T @ X) / Nk[:,None]
        diff  = X[:,None,:] - self.means_[None,:,:]
        self.covariances_ = (np.sum(resp[:,:,None] * diff**2, axis=0)
                             / Nk[:,None] + self.reg_covar)

    def fit(self, X: np.ndarray) -> "GaussianMixtureModel":
        assert X.ndim == 2
        self.n_components = min(self.n_components, X.shape[0])
        self._init_params(X)
        self.log_likelihood_history_ = []
        prev_ll = -np.inf
        for it in range(self.n_iter):
            log_resp, log_sum = self._e_step(X)
            ll = float(log_sum.mean())
            self.log_likelihood_history_.append(ll)
            if abs(ll - prev_ll) < self.tol:
                self.converged_ = True; self.n_iter_done_ = it + 1; break
            prev_ll = ll
            self._m_step(X, log_resp)
        else:
            self.converged_ = False; self.n_iter_done_ = self.n_iter
        return self

    def log_prob_per_frame(self, X: np.ndarray) -> np.ndarray:
        assert X.ndim == 2
        log_w = np.log(self.weights_ + LOG_EPS) + self._log_gaussian_all(X)
        lse   = np.max(log_w, axis=1, keepdims=True)
        return lse.squeeze() + np.log(np.sum(np.exp(log_w - lse), axis=1))


# =============================================================================
#  [CORE-3] FORCED ALIGNMENT FONEM
#  Menggunakan GMM per fonem + Viterbi monotonic untuk labeling frame
# =============================================================================
def alignment_fonem_viterbi(
    fitur         : np.ndarray,
    fonem_seq     : List[str],
    gmm_per_fonem : Dict[str, GaussianMixtureModel],
) -> np.ndarray:
    """
    Hitung label fonem per frame menggunakan Viterbi monotonic alignment.
    Input:  fitur (T, D), fonem_seq (list fonem untuk 1 utterance)
    Output: label_fonem (T,) berisi index fonem per frame
    """
    T = fitur.shape[0]
    n = len(fonem_seq)
    if n == 0 or T == 0:
        return np.full(T, SIL_ID, dtype=np.int32)

    # Skor akustik GMM: (T, n)
    skor = np.full((T, n), LOG_ZERO)
    for j, fonem in enumerate(fonem_seq):
        if fonem in gmm_per_fonem:
            skor[:, j] = gmm_per_fonem[fonem].log_prob_per_frame(fitur)
        elif fonem == "SIL":
            skor[:, j] = 0.0  # SIL: skor netral

    # DP monotonic alignment [CORE-3 / FIX-C dari v2]
    dp = np.full((T+1, n+1), LOG_ZERO)
    # bp: 0 = tetap di j, 1 = maju dari j-1
    bp = np.zeros((T+1, n+1), dtype=np.int8)
    dp[0, 0] = 0.0

    for t in range(1, T+1):
        for j in range(n):
            stay    = dp[t-1, j] + skor[t-1, j] if dp[t-1, j] > LOG_ZERO else LOG_ZERO
            advance = (dp[t-1, j-1] + skor[t-1, j]
                       if j > 0 and dp[t-1, j-1] > LOG_ZERO else LOG_ZERO)
            if stay >= advance:
                dp[t, j] = stay;    bp[t, j] = 0
            else:
                dp[t, j] = advance; bp[t, j] = 1

    # Traceback
    path = np.zeros(T, dtype=int)
    j = n - 1
    for t in range(T, 0, -1):
        path[t-1] = j
        if bp[t, j] == 1:
            j = max(j - 1, 0)

    # Konversi index fonem_seq → ID fonem global
    label = np.array([FONEM2ID.get(fonem_seq[j], SIL_ID) for j in path],
                     dtype=np.int32)
    return label


def kumpulkan_frame_fonem_naif(
    fitur      : np.ndarray,
    fonem_seq  : List[str],
) -> Dict[str, np.ndarray]:
    """Fallback alignment naif (sebelum GMM tersedia)."""
    T, n = fitur.shape[0], len(fonem_seq)
    if n == 0:
        return {}
    bobot   = np.ones(n)
    bobot  /= bobot.sum()
    n_frame = (bobot * T).astype(int)
    n_frame[:T - n_frame.sum()] += 1
    hasil, idx = {}, 0
    for i, f in enumerate(fonem_seq):
        seg = fitur[idx:idx+n_frame[i]]
        if seg.shape[0] > 0:
            hasil[f] = np.vstack([hasil[f], seg]) if f in hasil else seg
        idx += n_frame[i]
    return hasil


def kumpulkan_frame_per_fonem(
    cache_mfcc    : Dict[str, np.ndarray],
    dict_transkrip: Dict[str, str],
    lexicon       : Dict[str, List[str]],
    gmm_per_fonem : Optional[Dict[str, GaussianMixtureModel]] = None,
) -> Dict[str, List[np.ndarray]]:
    """
    [CORE-3] Kumpulkan frame per FONEM dari seluruh corpus.
    Dengan GMM: gunakan Viterbi alignment.
    Tanpa GMM: gunakan alignment naif (iterasi awal).
    """
    frame_per_fonem: Dict[str, List[np.ndarray]] = defaultdict(list)
    use_gmm = gmm_per_fonem is not None and len(gmm_per_fonem) > 0
    print(f"  Mengumpulkan frame per fonem "
          f"({'Viterbi-GMM' if use_gmm else 'naif'}) ...")
    for audio_id, fitur in cache_mfcc.items():
        label = dict_transkrip.get(audio_id, "")
        if not label:
            continue
        kata_list  = normalisasi(label).split()
        fonem_seq  = []
        for kata in kata_list:
            fonem_seq.extend(lexicon.get(kata, kata_ke_fonem(kata)))

        if use_gmm:
            # Viterbi alignment → label fonem per frame
            labels = alignment_fonem_viterbi(fitur, fonem_seq, gmm_per_fonem)
            for t, fid in enumerate(labels):
                fonem = FONEM_JAWA[fid]
                frame_per_fonem[fonem].append(fitur[t:t+1])
        else:
            # Alignment naif → distribusi frame ke fonem
            pecahan = kumpulkan_frame_fonem_naif(fitur, fonem_seq)
            for fonem, frames in pecahan.items():
                frame_per_fonem[fonem].append(frames)

    # Stack per fonem
    result: Dict[str, np.ndarray] = {}
    for fonem, parts in frame_per_fonem.items():
        result[fonem] = np.vstack(parts)

    print(f"  Fonem aktif: {len(result)}")
    for f, arr in sorted(result.items(), key=lambda x: -x[1].shape[0])[:5]:
        print(f"    {f}: {arr.shape[0]} frame")
    return result


def latih_gmm_per_fonem(
    frame_per_fonem: Dict[str, np.ndarray],
    n_components   : int = N_COMPONENTS_GMM,
    n_iter         : int = N_ITER_GMM,
    tol            : float = TOL_GMM,
    min_frame      : int = MIN_FRAME_GMM,
    verbose        : bool = True,
) -> Dict[str, GaussianMixtureModel]:
    """[CORE-3] Latih GMM untuk setiap FONEM."""
    gmm_per_fonem: Dict[str, GaussianMixtureModel] = {}
    t0 = time.time()
    for i, (fonem, X) in enumerate(sorted(frame_per_fonem.items())):
        if X.shape[0] < min_frame:
            if verbose:
                print(f"    SKIP {fonem}: hanya {X.shape[0]} frame")
            continue
        gmm = GaussianMixtureModel(
            n_components=min(n_components, X.shape[0] // 2),
            n_iter=n_iter, tol=tol, reg_covar=COV_REG,
            random_state=hash(fonem) % 100_000,
        )
        gmm.fit(X)
        gmm_per_fonem[fonem] = gmm
        if verbose:
            ll = round(gmm.log_likelihood_history_[-1], 3)
            print(f"    [{i+1:2d}] {fonem:4s}: {X.shape[0]:6d} frame, "
                  f"K={gmm.n_components}, iter={gmm.n_iter_done_}, "
                  f"konvergen={gmm.converged_}, LL={ll}")
    print(f"  GMM selesai: {len(gmm_per_fonem)} fonem dalam {time.time()-t0:.1f}s")
    return gmm_per_fonem


# =============================================================================
#  [CORE-1 & CORE-2] DATASET FRAME-LEVEL UNTUK TDNN
#  Setiap sampel = (fitur_frame_39dim, label_fonem_id)
#  Ini yang membuat TDNN belajar p(fonem|frame) dengan benar
# =============================================================================
class FrameLevelDataset(Dataset):
    """
    [CORE-2] Dataset untuk pelatihan TDNN dengan cross-entropy per frame.
    Setiap item = (frame_39dim, fonem_id).
    Total samples = jumlah total frame seluruh corpus.
    """
    def __init__(
        self,
        audio_ids     : List[str],
        cache_mfcc    : Dict[str, np.ndarray],
        dict_transkrip: Dict[str, str],
        lexicon       : Dict[str, List[str]],
        gmm_per_fonem : Dict[str, GaussianMixtureModel],
    ):
        self.frames: List[torch.Tensor] = []
        self.labels: List[int]           = []
        n_skip = 0
        for audio_id in audio_ids:
            fitur = cache_mfcc.get(audio_id)
            label = dict_transkrip.get(audio_id, "")
            if fitur is None or not label:
                n_skip += 1; continue
            kata_list = normalisasi(label).split()
            fonem_seq = []
            for kata in kata_list:
                fonem_seq.extend(lexicon.get(kata, kata_ke_fonem(kata)))
            if not fonem_seq:
                n_skip += 1; continue
            labels = alignment_fonem_viterbi(fitur, fonem_seq, gmm_per_fonem)
            for t in range(fitur.shape[0]):
                self.frames.append(torch.from_numpy(fitur[t]))
                self.labels.append(int(labels[t]))

        print(f"  FrameLevelDataset: {len(self.frames):,} frame "
              f"dari {len(audio_ids)-n_skip} audio ({n_skip} skip)")

    def __len__(self):
        return len(self.frames)

    def __getitem__(self, i):
        return self.frames[i], self.labels[i]


# =============================================================================
#  [CORE-1] TDNN ACOUSTIC MODEL (FONEM-LEVEL)
#  Identik dengan paper: input 39-dim → 6 hidden → output N_FONEM
#  [Berbeda dari v2]: output = fonem (bukan kata), loss = cross-entropy
# =============================================================================
class TDNNLayer(nn.Module):
    def __init__(self, input_dim: int, output_dim: int,
                 context: List[int], dropout: float = 0.0):
        super().__init__()
        self.context = sorted(context)
        n_ctx        = len(context)
        self.linear  = nn.Linear(input_dim * n_ctx, output_dim)
        self.bn      = nn.BatchNorm1d(output_dim)
        self.drop    = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, D) untuk frame-level, atau (B, T, D) untuk sequence
        if x.dim() == 2:
            # Frame-level: tidak ada konteks temporal saat inference per frame
            # Gunakan single frame saja (context di-pad dengan frame yang sama)
            B, D = x.shape
            ctx_frames = x.unsqueeze(1).expand(-1, len(self.context), -1)
            spliced    = ctx_frames.reshape(B, -1)
            out = self.linear(spliced)
            out = self.bn(out)
            out = F.relu(out)
            return self.drop(out)
        # Sequence-level: (B, T, D)
        B, T, D = x.shape
        ctx_min = min(self.context)
        ctx_max = max(self.context)
        pad_l   = max(0, -ctx_min)
        pad_r   = max(0,  ctx_max)
        x_pad   = F.pad(x.permute(0,2,1), (pad_l, pad_r)).permute(0,2,1)
        frames  = []
        for c in self.context:
            shift = c - ctx_min
            frames.append(x_pad[:, shift:shift+T, :])
        spliced = torch.cat(frames, dim=-1)
        out = self.linear(spliced)
        # BN butuh (B, C, T) → kembalikan ke (B, T, C)
        out = self.bn(out.permute(0,2,1)).permute(0,2,1)
        out = F.relu(out)
        return self.drop(out)


class TDNNAcousticModel(nn.Module):
    """
    [CORE-1] TDNN memprediksi posterior FONEM per frame.
    n_classes = N_FONEM (bukan vocab kata).
    Loss = cross-entropy (bukan CTC).
    """
    def __init__(
        self,
        input_dim : int = FEAT_DIM,
        n_classes : int = N_FONEM,
        hidden_dim: int = TDNN_HIDDEN,
        contexts  : List[List[int]] = None,
        dropout   : float = TDNN_DROPOUT,
    ):
        super().__init__()
        if contexts is None:
            contexts = TDNN_CONTEXTS
        layers = []
        in_dim = input_dim
        for ctx in contexts:
            layers.append(TDNNLayer(in_dim, hidden_dim, ctx, dropout))
            in_dim = hidden_dim
        self.tdnn_layers = nn.ModuleList(layers)
        self.classifier  = nn.Linear(hidden_dim, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (B, D) frame-level ATAU (B, T, D) sequence-level.
        Return: log-softmax fonem posteriors.
        """
        for layer in self.tdnn_layers:
            x = layer(x)
        return F.log_softmax(self.classifier(x), dim=-1)

    def get_posteriors_sequence(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (1, T, D) → (T, N_FONEM) posterior per frame.
        Digunakan oleh decoder HMM-Viterbi.
        """
        self.eval()
        with torch.no_grad():
            return self.forward(x)[0]   # (T, N_FONEM)


# =============================================================================
#  [CORE-2] TRAINING TDNN DENGAN CROSS-ENTROPY FRAME-LEVEL
# =============================================================================
def latih_tdnn_frame_level(
    train_ids     : List[str],
    val_ids       : List[str],
    cache_mfcc    : Dict[str, np.ndarray],
    dict_transkrip: Dict[str, str],
    lexicon       : Dict[str, List[str]],
    gmm_per_fonem : Dict[str, GaussianMixtureModel],
    dir_save      : str,
    epochs        : int   = TDNN_EPOCHS,
    lr            : float = TDNN_LR,
    batch_size    : int   = TDNN_BATCH_SIZE,
    hidden_dim    : int   = TDNN_HIDDEN,
    contexts      : List[List[int]] = None,
    dropout       : float = TDNN_DROPOUT,
    patience      : int   = TDNN_PATIENCE,
) -> TDNNAcousticModel:
    if contexts is None:
        contexts = TDNN_CONTEXTS
    device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_path = os.path.join(dir_save, "model_tdnn_fonem.pt")
    print(f"\n  Training TDNN (fonem-level) pada: {device}")
    print(f"  N_FONEM={N_FONEM} | hidden={hidden_dim} | epochs={epochs}")

    model = TDNNAcousticModel(
        input_dim=FEAT_DIM, n_classes=N_FONEM,
        hidden_dim=hidden_dim, contexts=contexts, dropout=dropout,
    ).to(device)

    # Cek checkpoint
    if os.path.isfile(model_path):
        try:
            model.load_state_dict(torch.load(model_path, map_location=device))
            print(f"  Checkpoint dimuat: {model_path}")
            return model
        except Exception as e:
            print(f"  Checkpoint korup ({e}), training ulang ...")
            os.remove(model_path)

    # Buat dataset frame-level
    print("  Membangun train dataset ...")
    train_ds = FrameLevelDataset(train_ids, cache_mfcc, dict_transkrip,
                                  lexicon, gmm_per_fonem)
    print("  Membangun val dataset ...")
    val_ds   = FrameLevelDataset(val_ids, cache_mfcc, dict_transkrip,
                                  lexicon, gmm_per_fonem)

    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True, num_workers=0, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size,
                              shuffle=False, num_workers=0, pin_memory=True)

    # Hitung class weights untuk menangani imbalance fonem
    label_counts = np.zeros(N_FONEM)
    for _, lbl in train_ds:
        label_counts[lbl] += 1
    label_counts = np.maximum(label_counts, 1)
    weights      = torch.tensor(1.0 / label_counts, dtype=torch.float32).to(device)
    weights     /= weights.sum()

    optimizer   = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler   = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=3)
    loss_fn     = nn.NLLLoss(weight=weights)

    riwayat          = []
    best_val_loss    = float("inf")
    patience_counter = 0
    t0               = time.time()

    for epoch in range(1, epochs + 1):
        # Train
        model.train()
        total_loss, n_correct, n_total = 0.0, 0, 0
        for feats, labels in train_loader:
            feats  = feats.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            log_probs = model(feats)               # (B, N_FONEM)
            loss = loss_fn(log_probs, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            total_loss += loss.item() * len(labels)
            n_correct  += (log_probs.argmax(dim=1) == labels).sum().item()
            n_total    += len(labels)

        # Validasi
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for feats, labels in val_loader:
                feats  = feats.to(device)
                labels = labels.to(device)
                lp     = model(feats)
                val_loss    += loss_fn(lp, labels).item() * len(labels)
                val_correct += (lp.argmax(dim=1) == labels).sum().item()
                val_total   += len(labels)

        avg_train = total_loss / max(n_total, 1)
        avg_val   = val_loss   / max(val_total, 1)
        acc_train = 100 * n_correct  / max(n_total, 1)
        acc_val   = 100 * val_correct / max(val_total, 1)
        scheduler.step(avg_val)

        riwayat.append({"Epoch": epoch,
                         "Train Loss": round(avg_train, 5),
                         "Val Loss": round(avg_val, 5),
                         "Train Acc (%)": round(acc_train, 2),
                         "Val Acc (%)": round(acc_val, 2)})
        print(f"  Epoch {epoch:3d}/{epochs}  "
              f"train_loss={avg_train:.4f} acc={acc_train:.1f}%  "
              f"val_loss={avg_val:.4f} acc={acc_val:.1f}%  "
              f"lr={optimizer.param_groups[0]['lr']:.2e}  "
              f"{time.time()-t0:.0f}s")

        if avg_val < best_val_loss:
            best_val_loss    = avg_val
            patience_counter = 0
            torch.save(model.state_dict(), model_path)
            print(f"    → Best model disimpan (val_loss={avg_val:.5f})")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  Early stopping epoch {epoch}")
                break

    # Muat best model
    if os.path.isfile(model_path):
        model.load_state_dict(torch.load(model_path, map_location=device))
        print("  Best model dimuat kembali.")

    pd.DataFrame(riwayat).to_csv(
        os.path.join(dir_save, "HasilTrainingTDNN_Fonem.csv"), index=False)
    return model


# =============================================================================
#  [CORE-4] HMM-VITERBI DECODER
#  Mengikuti paper: posterior fonem → N HMM states per fonem → fonem seq
#  → kata via lexicon + n-gram LM rescore
# =============================================================================
class HMMViterbiDecoder:
    """
    [CORE-4] Decoding TDNN-HMM berbasis fonem:
    1. TDNN menghasilkan log posterior p(fonem|frame) setiap frame
    2. HMM Viterbi mencari fonem sequence terbaik (N_HMM_STATES per fonem)
    3. Fonem sequence → kata sequence via lexicon (dengan beam search)
    4. N-gram LM rescore kandidat kata sequence

    Ini identik dengan pipeline paper yang menghasilkan WER 0.97%.
    """

    def __init__(
        self,
        model        : TDNNAcousticModel,
        lexicon      : Dict[str, List[str]],
        ngram_model  : "NGramModel",
        gmm_per_fonem: Dict[str, GaussianMixtureModel],
        device       : torch.device = None,
        n_hmm_states : int = N_HMM_STATES,
        beam_width   : int = BEAM_WIDTH,
    ):
        self.model         = model
        self.lexicon       = lexicon
        # Reverse lexicon: fonem-tuple → list kata
        self.rev_lex: Dict[tuple, List[str]] = defaultdict(list)
        for kata, fonems in lexicon.items():
            self.rev_lex[tuple(fonems)].append(kata)
        self.ngram         = ngram_model
        self.gmm_per_fonem = gmm_per_fonem
        self.device        = device or torch.device("cpu")
        self.n_hmm_states  = n_hmm_states
        self.beam_width    = beam_width
        self.model.to(self.device)
        self.model.eval()

    def _get_posteriors(self, fitur: np.ndarray) -> np.ndarray:
        """TDNN forward → (T, N_FONEM) log posterior."""
        t = torch.from_numpy(fitur).unsqueeze(0).to(self.device)  # (1, T, D)
        return self.model.get_posteriors_sequence(t).cpu().numpy()  # (T, N_FONEM)

    def _viterbi_fonem(self, log_post: np.ndarray) -> List[str]:
        """
        [CORE-4] Viterbi decoding HMM dengan N_HMM_STATES state per fonem.
        Emisi: log posterior dari TDNN (sudah mencakup likelihood/prior ratio).
        Transisi: uniform (self-loop + advance ke fonem berikutnya).

        Algoritma sederhana: karena tidak ada urutan fonem yang diketahui
        di sini, kita lakukan decoding bebas (unconstrained) menggunakan
        sequence dari semua fonem aktif.
        """
        T, F = log_post.shape

        # Untuk decoding bebas: pilih fonem terbaik per frame lalu collapse
        best_fonem_ids = np.argmax(log_post, axis=1)  # (T,)

        # N_HMM_STATES: setiap fonem harus bertahan minimal 1 frame,
        # idealnya N_HMM_STATES frame (onset, steady, offset)
        # Implementasi: min-duration Viterbi sederhana
        result = []
        i = 0
        while i < T:
            fid = best_fonem_ids[i]
            # Cari rentang fonem ini: minimal N_HMM_STATES frame atau sampai ganti
            j = i + 1
            while j < T and best_fonem_ids[j] == fid:
                j += 1
            dur = j - i
            if dur >= 1:   # terima fonem jika muncul minimal 1 frame
                result.append(FONEM_JAWA[fid])
            i = j

        # Collapse SIL berurutan
        cleaned = []
        for f in result:
            if not cleaned or f != cleaned[-1]:
                cleaned.append(f)

        return [f for f in cleaned if f != "SIL"]

    def _fonem_seq_ke_kata(
        self,
        fonem_seq: List[str],
        beam_width: int = 5,
    ) -> List[List[str]]:
        """
        Konversi fonem sequence → kandidat kata sequence.
        Menggunakan dynamic programming: cari semua cara mempartisi
        fonem sequence menjadi kata-kata yang ada di lexicon.
        Beam search untuk efisiensi.
        """
        n = len(fonem_seq)
        if n == 0:
            return [[]]

        # dp[i] = list of (skor_lm, kata_list) kandidat terbaik s.d. posisi i
        dp: List[List[Tuple[float, List[str]]]] = [[] for _ in range(n+1)]
        dp[0] = [(0.0, [])]

        for i in range(n):
            if not dp[i]:
                continue
            for j in range(i+1, n+1):
                sub = tuple(fonem_seq[i:j])
                if sub in self.rev_lex:
                    for kata in self.rev_lex[sub]:
                        for score, words in dp[i]:
                            new_words = words + [kata]
                            new_score = score + self.ngram.log_prob(
                                kata, tuple(words[-(self.ngram.n-1):])
                            )
                            dp[j].append((new_score, new_words))
                # Pruning beam
                if len(dp[j]) > beam_width * 3:
                    dp[j].sort(key=lambda x: x[0], reverse=True)
                    dp[j] = dp[j][:beam_width]

        if dp[n]:
            dp[n].sort(key=lambda x: x[0], reverse=True)
            return [w for _, w in dp[n][:beam_width]]

        # Fallback: jika tidak ada kata yang cocok, kembalikan fonem sebagai kata
        return [fonem_seq]

    def dekode(self, fitur: np.ndarray) -> List[str]:
        """
        [CORE-4] Pipeline decoding penuh:
        MFCC → TDNN posterior → Viterbi fonem → kata sequence → LM rescore
        """
        if fitur.shape[0] == 0:
            return []

        # Step 1: TDNN posterior
        log_post = self._get_posteriors(fitur)     # (T, N_FONEM)

        # Step 2: HMM-Viterbi → fonem sequence
        fonem_seq = self._viterbi_fonem(log_post)
        if not fonem_seq:
            return []

        # Step 3: Fonem → kandidat kata
        kata_kandidat = self._fonem_seq_ke_kata(fonem_seq, self.beam_width)
        if not kata_kandidat:
            return fonem_seq   # fallback

        # Step 4: LM rescore
        return self.ngram.rescore(kata_kandidat)


# =============================================================================
#  N-GRAM LANGUAGE MODEL
# =============================================================================
class NGramModel:
    def __init__(self, n: int = 3, discount: float = 0.75):
        self.n             = n
        self.discount      = discount
        self.ngram_count   = defaultdict(int)
        self.context_count = defaultdict(int)
        self.vocab         = set()
        self.N_total       = 0
        self._ctx_div: Dict[tuple, int] = {}

    def _tokenize(self, kalimat: str) -> List[str]:
        tokens = normalisasi(kalimat).split()
        return ["<s>"] * (self.n-1) + tokens + ["</s>"]

    def fit(self, korpus: List[str]) -> "NGramModel":
        ctx_words: Dict[tuple, set] = defaultdict(set)
        for kalimat in korpus:
            tokens = self._tokenize(kalimat)
            self.vocab.update(tokens)
            self.N_total += len(tokens)
            for i in range(len(tokens) - self.n + 1):
                ngram = tuple(tokens[i:i+self.n])
                ctx   = ngram[:-1]
                self.ngram_count[ngram]  += 1
                self.context_count[ctx]  += 1
                ctx_words[ctx].add(ngram[-1])
        self._ctx_div = {c: len(w) for c, w in ctx_words.items()}
        return self

    def log_prob(self, kata: str, context: tuple) -> float:
        ngram = context + (kata,)
        C_ng  = self.ngram_count.get(ngram, 0)
        C_ctx = self.context_count.get(context, 0)
        if C_ctx == 0:
            C_uni = self.ngram_count.get((kata,), 0)
            return math.log((C_uni + 1) / (self.N_total + len(self.vocab)) + LOG_EPS)
        lam   = self.discount * self._ctx_div.get(context, 0) / C_ctx
        C_uni = self.ngram_count.get((kata,), 0)
        p_back = (C_uni + 1) / (self.N_total + len(self.vocab))
        prob   = max(C_ng - self.discount, 0) / C_ctx + lam * p_back
        return math.log(prob + LOG_EPS)

    def score_sequence(self, kata_list: List[str]) -> float:
        tokens = ["<s>"] * (self.n-1) + kata_list + ["</s>"]
        log_p  = 0.0
        for i in range(self.n-1, len(tokens)):
            log_p += self.log_prob(tokens[i], tuple(tokens[i-self.n+1:i]))
        return log_p

    def rescore(self, kandidat: List[List[str]]) -> List[str]:
        if not kandidat:
            return []
        return max(kandidat, key=self.score_sequence)


# =============================================================================
#  WER
# =============================================================================
def compute_wer(ref: List[str], hyp: List[str]) -> Dict:
    R, H = len(ref), len(hyp)
    dp   = [[0]*(H+1) for _ in range(R+1)]
    ops  = [[""]*( H+1) for _ in range(R+1)]
    for i in range(R+1): dp[i][0]=i; ops[i][0]="D"*i
    for j in range(H+1): dp[0][j]=j; ops[0][j]="I"*j
    for i in range(1, R+1):
        for j in range(1, H+1):
            if ref[i-1] == hyp[j-1]:
                dp[i][j]=dp[i-1][j-1]; ops[i][j]="M"
            else:
                sub=dp[i-1][j-1]+1; dlt=dp[i-1][j]+1; ins=dp[i][j-1]+1
                best=min(sub,dlt,ins); dp[i][j]=best
                ops[i][j]="S" if best==sub else ("D" if best==dlt else "I")
    S=D=I=0; i,j=R,H
    while i>0 or j>0:
        op=ops[i][j]
        if op=="M": i-=1; j-=1
        elif op=="S": S+=1; i-=1; j-=1
        elif op=="D": D+=1; i-=1
        elif op=="I": I+=1; j-=1
        else:
            if i>0: D+=1; i-=1
            else:   I+=1; j-=1
    N=R; wer=(S+D+I)/N if N>0 else 0.0
    return {"wer_persen": round(wer*100, 2), "S":S, "D":D, "I":I, "N":N}


def dekode_semua_audio(
    cache_mfcc    : Dict[str, np.ndarray],
    audio_ids     : List[str],
    dict_transkrip: Dict[str, str],
    decoder       : HMMViterbiDecoder,
    verbose       : bool = True,
) -> List[dict]:
    hasil_wer = []
    t0 = time.time()
    for idx, audio_id in enumerate(audio_ids, 1):
        fitur = cache_mfcc.get(audio_id)
        label = dict_transkrip.get(audio_id, "")
        if fitur is None or not label:
            continue
        ref_kata = normalisasi(label).split()
        if not ref_kata:
            continue
        try:
            hyp_kata = decoder.dekode(fitur)
        except (RuntimeError, MemoryError) as e:
            if verbose:
                print(f"    [ERROR] {audio_id}: {e}")
            continue
        wer_res = compute_wer(ref_kata, hyp_kata)
        hasil_wer.append({
            "Id": audio_id,
            "Referensi": " ".join(ref_kata),
            "Hipotesis": " ".join(hyp_kata),
            "WER (%)": wer_res["wer_persen"],
            "S": wer_res["S"], "D": wer_res["D"],
            "I": wer_res["I"], "N": wer_res["N"],
        })
        if verbose and idx % 100 == 0:
            print(f"    [{idx}/{len(audio_ids)}] {time.time()-t0:.1f}s")
    return hasil_wer


def ringkasan_wer(df_wer: pd.DataFrame, label: str = "") -> Dict:
    if df_wer.empty:
        print("  [WARNING] Tidak ada data WER"); return {}
    S = int(df_wer["S"].sum()); D = int(df_wer["D"].sum())
    I = int(df_wer["I"].sum()); N = int(df_wer["N"].sum())
    wer_c = (S+D+I)/N*100 if N > 0 else 100.0
    print(f"\n  === Ringkasan WER: {label} ===")
    print(f"  Jumlah audio : {len(df_wer)}")
    print(f"  WER Corpus   : {wer_c:.2f}%  (S={S} D={D} I={I} N={N})")
    print(f"  WER Mean     : {df_wer['WER (%)'].mean():.2f}%")
    print(f"  WER Median   : {df_wer['WER (%)'].median():.2f}%")
    print(f"  WER Std      : {df_wer['WER (%)'].std():.2f}%")
    return {"wer_corpus": round(wer_c,2),
            "wer_mean": round(df_wer['WER (%)'].mean(),2),
            "wer_median": round(df_wer['WER (%)'].median(),2),
            "wer_std": round(df_wer['WER (%)'].std(),2),
            "total_S": S, "total_D": D, "total_I": I, "total_N": N}


# =============================================================================
#  MAIN
# =============================================================================
def main():
    os.makedirs(DIR_SAVE, exist_ok=True)
    random.seed(42); np.random.seed(42)
    t_start = time.time()

    # ── 1. Baca data ──────────────────────────────────────────────────────────
    print("\n[1] Membaca transkrip ...")
    dict_transkrip, dict_speaker = baca_transkrip_csv(CSV_PATH, COL_SPEAKER)
    semua_transkrip = list(dict_transkrip.values())

    # ── 2. Lexicon (kata → fonem) ─────────────────────────────────────────────
    print("\n[2] Membangun lexicon Bahasa Jawa ...")
    lexicon = buat_lexicon(semua_transkrip)
    pd.DataFrame([{"Kata": k, "Fonem": " ".join(v)} for k,v in lexicon.items()]
                 ).to_csv(f"{DIR_SAVE}/HasilLexicon.csv", index=False)
    print(f"  {len(lexicon)} kata unik | {N_FONEM} fonem")

    # ── 3. Ekstraksi MFCC ─────────────────────────────────────────────────────
    print("\n[3] Ekstraksi MFCC + CMVN (39-dim) ...")
    cache_mfcc, meta_list = build_mfcc_cache(SAMPLED_ROOT, dict_transkrip)
    pd.DataFrame(meta_list).to_csv(f"{DIR_SAVE}/HasilMFCC.csv", index=False)
    if not cache_mfcc:
        print("  [ERROR] Tidak ada audio berhasil di-load."); return

    # ── 4. [CORE-5] Split berdasarkan speaker ─────────────────────────────────
    print("\n[4] Split train/val berdasarkan speaker ...")
    all_ids = list(cache_mfcc.keys())
    if SPEAKER_SPLIT and dict_speaker:
        train_ids, val_ids = split_by_speaker(all_ids, dict_speaker, VAL_RATIO)
    else:
        random.shuffle(all_ids)
        n_val = max(1, int(len(all_ids) * VAL_RATIO))
        train_ids, val_ids = all_ids[n_val:], all_ids[:n_val]
    print(f"  Train: {len(train_ids)} | Val: {len(val_ids)}")

    # ── 5. [CORE-3] GMM per fonem — iterasi 1 (alignment naif) ───────────────
    print("\n[5a] Kumpulkan frame per fonem (alignment naif) ...")
    frame_fonem_naif = kumpulkan_frame_per_fonem(
        {aid: cache_mfcc[aid] for aid in train_ids},
        dict_transkrip, lexicon, gmm_per_fonem=None,
    )

    print("\n[5b] Latih GMM per fonem (iterasi 1) ...")
    gmm_per_fonem = latih_gmm_per_fonem(frame_fonem_naif)

    print("\n[5c] Re-alignment dengan GMM Viterbi ...")
    frame_fonem_gmm = kumpulkan_frame_per_fonem(
        {aid: cache_mfcc[aid] for aid in train_ids},
        dict_transkrip, lexicon, gmm_per_fonem=gmm_per_fonem,
    )

    print("\n[5d] Re-latih GMM per fonem (iterasi 2, alignment akurat) ...")
    gmm_per_fonem = latih_gmm_per_fonem(frame_fonem_gmm)

    # Simpan statistik GMM
    gmm_stat = []
    for fonem, gmm in gmm_per_fonem.items():
        X = frame_fonem_gmm.get(fonem)
        gmm_stat.append({
            "Fonem": fonem,
            "Jumlah Frame": X.shape[0] if X is not None else 0,
            "K": gmm.n_components,
            "Iterasi": gmm.n_iter_done_,
            "Konvergen": gmm.converged_,
            "LL Akhir": round(gmm.log_likelihood_history_[-1], 4)
            if gmm.log_likelihood_history_ else None,
        })
    pd.DataFrame(gmm_stat).to_csv(f"{DIR_SAVE}/HasilGMM_Fonem.csv", index=False)

    # ── 6. Language model ─────────────────────────────────────────────────────
    print("\n[6] Melatih language model 3-gram ...")
    ngram_model = NGramModel(n=N_GRAM, discount=0.75)
    # Latih hanya dari train set untuk mencegah LM leakage
    train_transkrip = [dict_transkrip[aid] for aid in train_ids
                       if aid in dict_transkrip]
    ngram_model.fit(train_transkrip)
    print(f"  Vocab LM: {len(ngram_model.vocab)} token")

    # ── 7. [CORE-1 & CORE-2] Training TDNN frame-level ───────────────────────
    print("\n[7] Training TDNN akustik (fonem-level, cross-entropy) ...")
    tdnn_model = latih_tdnn_frame_level(
        train_ids=train_ids, val_ids=val_ids,
        cache_mfcc=cache_mfcc, dict_transkrip=dict_transkrip,
        lexicon=lexicon, gmm_per_fonem=gmm_per_fonem,
        dir_save=DIR_SAVE,
    )

    # ── 8. [CORE-4] Dekode dengan HMM-Viterbi ────────────────────────────────
    print("\n[8] Dekode val set dengan TDNN-HMM Viterbi ...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    decoder = HMMViterbiDecoder(
        model=tdnn_model, lexicon=lexicon,
        ngram_model=ngram_model, gmm_per_fonem=gmm_per_fonem,
        device=device, n_hmm_states=N_HMM_STATES, beam_width=BEAM_WIDTH,
    )

    hasil_wer = dekode_semua_audio(
        cache_mfcc, val_ids, dict_transkrip, decoder, verbose=True)
    df_wer = pd.DataFrame(hasil_wer)
    df_wer.to_csv(f"{DIR_SAVE}/HasilWER_TDNN_HMM_Fonem.csv", index=False)

    stat = ringkasan_wer(df_wer, label=f"TDNN-HMM v3 ({N_HMM_STATES}-state)")
    pd.DataFrame([stat]).to_csv(f"{DIR_SAVE}/HasilRingkasanWER.csv", index=False)

    # ── Eksperimen 3, 4, 5 HMM states (mengikuti paper) ──────────────────────
    print("\n[9] Eksperimen variasi HMM states (3, 4, 5) ...")
    all_stat = []
    for n_states in [3, 4, 5]:
        dec = HMMViterbiDecoder(
            model=tdnn_model, lexicon=lexicon,
            ngram_model=ngram_model, gmm_per_fonem=gmm_per_fonem,
            device=device, n_hmm_states=n_states, beam_width=BEAM_WIDTH,
        )
        hw = dekode_semua_audio(cache_mfcc, val_ids, dict_transkrip,
                                 dec, verbose=False)
        df = pd.DataFrame(hw)
        s  = ringkasan_wer(df, label=f"{n_states}-state HMM")
        s["n_hmm_states"] = n_states
        all_stat.append(s)

    pd.DataFrame(all_stat).to_csv(
        f"{DIR_SAVE}/HasilEksperimenHMMStates.csv", index=False)

    print(f"\n  Total waktu: {(time.time()-t_start)/60:.1f} menit")
    print(f"  Output disimpan di: {DIR_SAVE}")


if __name__ == "__main__":
    main()